# Email Processing Downstream EDA

## Purpose
Traces every downstream DataFrame transformation for the `email_processing()` output from the `Transactions` class in `data_processing.py`, as consumed by `data_processing_promotion.py`.

---

## Notebook Structure

| Cell | Stage | Description |
|------|-------|-------------|
| 1 | **Setup** | Snowflake connection + direct source queries (no UC intermediate tables) |
| 2 | **Setup** | Utility functions, classes (all inlined — zero external .py dependencies) |
| 3 | **Setup** | Imports, Spark session, config params, load flat files |
| 2 | **Step 1 — Raw Email Source** | Load `oe_kerendia_fact_npp_inference` table, show schema + row count |
| 3 | **Step 2 — email_processing() Output** | Run email_processing() standalone, show resulting IDC DataFrame |
| 4 | **Step 3 — Concat into proto_idc** | Show email rows after `pd.concat([calls, emails, events])` |
| 5 | **Step 4 — get_transaction_idc() Post-Processing** | Date lag (+7 days), YRMO filter, standardize, nan_handler, dedup |
| 6 | **Step 5 — get_counts() Pivot** | R/I/E metric pivots + content counts, merged on CUSTOMER_ID × YRMO |
| 7 | **Step 6 — SequenceProcessor Output** | Sequences, channel/content counts, gap-group sequences |
| 8 | **Step 7 — process_timeframes() Aggregation** | C1/C3/P3/P6/P12 window sums + touchpoint rollups |
| 9 | **Step 8 — Final MDB Assembly** | Merge sequences + counts + demographics + CVI + sales extensions |

---

## Input Sources

### Unity Catalog Tables
| Table | Used In |
|-------|---------|
| `ph-com-ai-us-prod-catalog-2553213839599380.kerendia_oe_journey.oe_kerendia_fact_npp_inference` | `email_processing()` — primary NPP facts |
| `ph-com-ai-us-prod-catalog-2553213839599380.kerendia_oe_journey.oe_kerendia_fact_call_inference` | `calls_processing()` — call facts |
| `ph-com-ai-us-prod-catalog-2553213839599380.kerendia_oe_journey.oe_kerendia_fact_evnt_prsnl_prom_inference` | `pp_processing()` — personal promo events |
| `ph-com-ai-us-prod-catalog-2553213839599380.kerendia_oe_journey.oe_bayer_modeling_nbrx_final_inference` | `get_cvi_dataset()` — NBRx modeling |
| `ph-com-ai-us-prod-catalog-2553213839599380.kerendia_oe_journey.blink_kerendia_nbrx_rj_rv_pd_inference` | `get_cvi_dataset()` — BlinkRx coverage |
| `ph-com-ai-us-prod-catalog-2553213839599380.kerendia_oe_journey.nba_oe_pd_rj_rv_trx_inference` | `get_cvi_dataset()` — TRx inference |

### Delta Volume Paths
| Path | Used In |
|------|---------|
| `.../oe_volume_space/notebook_output/13_5_oe_modelling_universe_final` | `df_target` — modelling universe (flag=mdb) |
| `.../oe_volume_space/notebook_output/temp_kerendia_hcp_attr` | `get_demog()` — HCP demographic attributes |
| `.../oe_volume_space/notebook_output/customer_segment` | Customer segment OHE in final MDB |

### Flat Files (Excel)
| File | Used In |
|------|---------|
| `5_5_2026_Bayer_OE_updated_channel_eng_exp_flag_latest.xlsx` | `df_flag_new` — Exposure/Engagement/Acted flag lookup |
| `Content_tagging_6_5.xlsx` | `df_tagg` — Content category tagging for NPP joins |
| `mdb_prep_config.xlsx` | `date_ranges` + `metric_mapping` (Date Ranges & Metric Mappings sheets) |

---

## Output Storage
The final MDB is written as Delta to:
```
/Volumes/ph-com-ai-us-prod-catalog-2553213839599380/oe_journey/oe_volume_space/notebook_output/model_training_mdb_v6_p12_added
```
Also saved as a temporary Parquet via `save_to_s3()`.

In [0]:
%pip install openpyxl statsmodels --quiet

In [0]:
%run "/Workspace/Users/wenjie.chen@bayer.com/OE/Kerendia_OE_Inference/00_Helper_Notebooks/00_connection_inference"

In [0]:
# =========================================================================
# Direct Snowflake queries → saved as parquet on UC Volume
# Source: Promotional_data_ingestion_layer notebook
# Set REFRESH = True to re-pull from Snowflake; False to read cached parquet
# =========================================================================
import os

VOLUME_BASE = "/Volumes/ph-com-ai-us-prod-catalog-2553213839599380/kerendia_nba_3/kerendia_nba_3_1_volume/hcp_level_model/dev_pipeline/query_output"
REFRESH = False  # <-- flip to True when you need a fresh Snowflake pull

# --- Table 1: FACT_CALL ---
call_path = f"{VOLUME_BASE}/oe_kerendia_fact_call.parquet"
if REFRESH or not os.path.exists(call_path):
    print("Pulling CALL from Snowflake...")
    df_call_src = Get_Data_Snowflakes("""
        SELECT DISTINCT 
            B.BAYER_CUST_ID AS BHID,
            CASE 
                WHEN UPPER(A.CALL_REC_TYP_NM) LIKE '%BPH_FACE2FACE_SIG%' OR UPPER(A.CALL_REC_TYP_NM) LIKE 'FACE2FACE_SIG%' THEN 'F2F'
                WHEN UPPER(A.CALL_REC_TYP_NM) LIKE '%BPH_NON_FACE2FACE%' THEN 'PHONE'
                WHEN UPPER(A.CALL_REC_TYP_NM) IN ('BPH_GROUP_CALL_SIG', 'GROUP CALL SIGNED', 'EVENT_VOD') THEN 'GROUP_CALL'
                WHEN UPPER(A.CALL_REC_TYP_NM) LIKE '%BPH_REMOTE%' THEN 'REMOTE_CALL'
                ELSE A.CALL_REC_TYP_NM 
            END AS CHNNL_NM,
            A.DATE_ID,
            A.CALL_ID,
            A.CALL_TYP,
            A.DETL_PROD
        FROM CPH_DB_PROD.MODEL_V2.FACT_CALL A
        LEFT JOIN (
            SELECT SOURCE_CUST_ID, BAYER_CUST_ID 
            FROM CPH_DB_PROD.MODEL_V2.MAP_CUSTOMER_XREF
        ) B
        ON A.SOURCE_CUST_ID = B.SOURCE_CUST_ID
        WHERE UPPER(A.STAT) = 'SUBMITTED_VOD'
            AND UPPER(A.DETL_PROD) = 'KERENDIA'
            AND B.BAYER_CUST_ID NOT LIKE '%BHO%'
            AND B.BAYER_CUST_ID IS NOT NULL
            AND A.DATE_ID >= TO_CHAR(DATEADD('month', -24, CURRENT_DATE()), 'YYYYMMDD')
    """)
    df_call_src.write.mode("overwrite").parquet(call_path)
    print(f"  Saved to {call_path}")
df_call_src = spark.read.parquet(call_path)
print(f"CALL rows: {df_call_src.count():,}  ({call_path})")

# --- Table 2: NPP ---
npp_path = f"{VOLUME_BASE}/oe_kerendia_fact_npp.parquet"
if REFRESH or not os.path.exists(npp_path):
    print("Pulling NPP from Snowflake...")
    npp_columns = [
        'CUST_ID', 'PROD_ID', 'LITCODE', 'DATE_ID', 'CHNL_ID', 'MTR_NM', 'VEND_NM',
        'CHNL_NM', 'RPT_CHNL', 'CPGN_NM', 'TACT_TYP', 'HCP_SEG_1', 'CHNL_TYP',
        'CPGN_AUDS', 'EMAIL_SUBJ_LN', 'CRTV_NM', 'TACT_NM', 'PRMRY_KEY_MSG',
        'COUNTRY_NM', 'HCP_PRMRY_SPCL', 'HCP_TYP', 'HCP_ID_TYP', 'CF_NAME',
        'CF_INDICATION', 'LINK_URL', 'TKN_ID', 'DEVC', 'VEND_PROD', 'BAYER_HCP_ID',
        'HCP_PRMRY_CITY', 'HCP_PRMRY_STATE', 'HCP_PRMRY_ZIP', 'HCP_PRMRY_STATE_LIC_NBR',
        'HCP_PAR_SPCL_GRP', 'PROD_ID_TYP', 'BAYER_PROD_ID', 'CNTT_NM', 'UNIQ_NPP_ID',
        'TRGT_FLAG', 'BUS_UNIT', 'PROD_BRAND_NM'
    ]
    df_npp_src = Get_Data_Snowflakes(f"""
        SELECT {', '.join(npp_columns)}
        FROM CPH_DB_PROD.ANALYTICS_V2.ANLT_BASE_FACT_NONPERSONAL_PROMOTIONS
        WHERE DATE_ID >= TO_CHAR(DATEADD('month', -24, CURRENT_DATE()), 'YYYYMMDD')
    """)
    df_npp_src.write.mode("overwrite").parquet(npp_path)
    print(f"  Saved to {npp_path}")
df_npp_src = spark.read.parquet(npp_path)
print(f"NPP rows:  {df_npp_src.count():,}  ({npp_path})")

# --- Table 3: Events ---
evnt_path = f"{VOLUME_BASE}/oe_kerendia_fact_evnt_prsnl_prom.parquet"
if REFRESH or not os.path.exists(evnt_path):
    print("Pulling EVNT from Snowflake...")
    evnt_columns = [
        'EVNT_ID', 'PROD_ID', 'DATE_ID', 'CUST_HCP_ID', 'EVNT_CAT',
        'EVNT_TYP', 'EVNT_SUB_TYP', 'ATND', 'EVNT_FMT', 'EVNT_TPIC_NM', 'PROD_BRAND_NM'
    ]
    df_evnt_src = Get_Data_Snowflakes(f"""
        SELECT {', '.join(evnt_columns)}
        FROM CPH_DB_PROD.ANALYTICS_V2.ANLT_BASE_FACT_EVNT_PRSNL_PROM
        WHERE DATE_ID >= TO_CHAR(DATEADD('month', -24, CURRENT_DATE()), 'YYYYMMDD')
    """)
    df_evnt_src.write.mode("overwrite").parquet(evnt_path)
    print(f"  Saved to {evnt_path}")
df_evnt_src = spark.read.parquet(evnt_path)
print(f"EVNT rows: {df_evnt_src.count():,}  ({evnt_path})")

print("\nAll 3 source tables ready (parquet-cached on Volume).")

In [0]:
# =========================================================================
# Utility functions inlined from handy_functions.py and utils.py
# =========================================================================
import re
import datetime
from itertools import product
from collections import Counter
import math, time
from typing import Dict, List

# --- handy_functions.py ---
def nan_handler(dataframe, rules={'drop': ["hcp_id", "exposed_on", "channel_id"], 'fill': {1: ['count']}}):
    if rules['drop']:
        dataframe = dataframe.dropna(axis=0, subset=rules['drop'])
    if rules['fill']:
        for key in rules['fill'].keys():
            dataframe[rules['fill'][key]] = dataframe[rules['fill'][key]].fillna(value=key, axis=0)
    dataframe.isna().any()
    return dataframe.copy(deep=True)

def standardize_columns(dataframe, columns=[]):
    pattern = re.compile('[a-zA-Z0-9]+')
    for column in columns:
        dataframe[column] = dataframe[column].apply(lambda value: "_".join(re.findall(pattern, str(value))).upper())
    return dataframe.copy()

def convert_to_numeric(dataframe, column):
    try:
        dataframe[column] = pd.to_numeric(dataframe[column])
        return dataframe.copy(deep=True)
    except ValueError:
        id_list, problem_list = [], []
        for index, val in dataframe.iterrows():
            value = val[column]
            try:
                id_list.append(int(value))
            except ValueError:
                id_list.append(value)
                problem_list.append(value)
        dataframe[column] = id_list
    print(problem_list)
    return dataframe.copy(deep=True)

# --- utils.py (content_tagging only) ---
def content_tagging(df, column_name, content_dict=None):
    def prioritize_content(value):
        if pd.isna(value) or not value.strip():
            return None
        elements = value.split('|')
        count_dict = {}
        for el in elements:
            count_dict[el] = count_dict.get(el, 0) + 1
        sorted_elements = sorted(count_dict.keys(), key=lambda x: (-count_dict[x], elements.index(x)))
        return sorted_elements[0] if sorted_elements else None
    df[column_name] = df[column_name].apply(prioritize_content)
    if content_dict is not None:
        df[column_name] = df[column_name].map(content_dict)
    return df[column_name]

# =========================================================================
# Sequence Generation Engine (inlined from Sequence_Generation_Engine.py)
# =========================================================================
class zs_pa_dataprep:
    def __init__(self, customer_id, channel_name_col, channel_number_col,
                 content_name_col, content_number_col, date_column):
        self.customer_id = customer_id
        self.channel_name_col = channel_name_col
        self.channel_number_col = channel_number_col
        self.content_name_col = content_name_col
        self.content_number_col = content_number_col
        self.date_column = date_column

    def journey_rollup(self, activity_data, start_date, end_date):
        activity_data = activity_data[
            (activity_data[self.date_column] >= start_date) &
            (activity_data[self.date_column] <= end_date)
        ]
        activity_data = activity_data.sort_values([self.customer_id, self.date_column], ascending=[True, True])
        activity_data['days_since_start'] = (activity_data[self.date_column] - start_date).dt.days
        customers = activity_data[self.customer_id].unique().tolist()
        num_periods = (end_date - start_date).days
        datelist = pd.date_range(start_date, periods=num_periods + 1).tolist()
        all_customer_dates = pd.DataFrame(list(product(customers, datelist)), columns=[self.customer_id, self.date_column])
        all_customer_dates[self.date_column] = pd.to_datetime(all_customer_dates[self.date_column])
        activity_data[self.date_column] = pd.to_datetime(activity_data[self.date_column])
        all_customer_dates = all_customer_dates.merge(activity_data, how='left', on=[self.customer_id, self.date_column], validate='many_to_many').reset_index()
        all_customer_dates[self.channel_name_col] = all_customer_dates[self.channel_name_col].fillna('GAP')
        all_customer_dates[self.channel_number_col] = all_customer_dates[self.channel_number_col].fillna(0)
        all_customer_dates[self.content_name_col] = all_customer_dates[self.content_name_col].fillna('GAP')
        all_customer_dates[self.content_number_col] = all_customer_dates[self.content_number_col].fillna(0)
        full_list_channel_name_seq, full_list_channel_code_seq = [], []
        list_channel_name_seq, list_channel_code_seq = [], []
        full_list_content_name_seq, full_list_content_code_seq = [], []
        list_content_name_seq, list_content_code_seq = [], []
        list_time_seq = []
        grouped = all_customer_dates.groupby(self.customer_id)
        grouped_2 = activity_data.groupby(self.customer_id)
        for i in all_customer_dates[self.customer_id].unique():
            full_list_channel_name_seq.append(list(grouped.get_group(i)[self.channel_name_col]))
            full_list_channel_code_seq.append(list(grouped.get_group(i)[self.channel_number_col]))
            list_channel_name_seq.append(list(grouped_2.get_group(i)[self.channel_name_col]))
            list_channel_code_seq.append(list(grouped_2.get_group(i)[self.channel_number_col]))
            full_list_content_name_seq.append(list(grouped.get_group(i)[self.content_name_col]))
            full_list_content_code_seq.append(list(grouped.get_group(i)[self.content_number_col]))
            list_content_name_seq.append(list(grouped_2.get_group(i)[self.content_name_col]))
            list_content_code_seq.append(list(grouped_2.get_group(i)[self.content_number_col]))
            time_seq = list(grouped_2.get_group(i)['days_since_start'])
            temp_time_seq = [(time_seq[j + 1] - time_seq[j]) for j in range(len(time_seq) - 1)]
            temp_time_seq.insert(0, time_seq[0])
            list_time_seq.append(temp_time_seq)
        return_data = pd.DataFrame()
        return_data[self.customer_id] = all_customer_dates[self.customer_id].unique()
        return_data['full_channelName_seq'] = full_list_channel_name_seq
        return_data['full_channelCode_seq'] = full_list_channel_code_seq
        return_data['channelName_seq'] = list_channel_name_seq
        return_data['channelCode_seq'] = list_channel_code_seq
        return_data['full_contentName_seq'] = full_list_content_name_seq
        return_data['full_contentCode_seq'] = full_list_content_code_seq
        return_data['contentName_seq'] = list_content_name_seq
        return_data['contentCode_seq'] = list_content_code_seq
        return_data['timeSeq'] = list_time_seq
        return_data = return_data.fillna(0)
        return return_data

    def channel_count(self, master_data, event_name, channel_list):
        channel_count_data = pd.DataFrame()
        channel_count_data[self.customer_id] = master_data[self.customer_id]
        for channel in channel_list:
            channel_count_data[channel] = [x.count(channel) for x in master_data[event_name]]
        return channel_count_data

    def content_count(self, master_data, event_name, content_list):
        content_count_data = pd.DataFrame()
        content_count_data[self.customer_id] = master_data[self.customer_id]
        for content in content_list:
            content_count_data[content] = [x.count(content) for x in master_data[event_name]]
        return content_count_data

print("Utility functions and zs_pa_dataprep class loaded (no external file dependency).")

In [0]:
# =========================================================================
# Transactions class (inlined from data_processing.py)
# =========================================================================
class Transactions:
    def __init__(self, start_yrmo, end_yrmo, product_key, country, metric_mapping, df_flag_new, df_target, df_tagg, df_call_src=None, df_npp_src=None, df_evnt_src=None, env='dev'):
        self.START_YRMO = start_yrmo
        self.END_YRMO = end_yrmo
        self.PRODUCT_KEY = product_key
        self.COUNTRY = country
        self.metric_mapping = metric_mapping
        self.df_flag_new = df_flag_new
        self.env = env
        self.df_target = df_target
        self.df_tagg = df_tagg
        self.df_call_src = df_call_src
        self.df_npp_src = df_npp_src
        self.df_evnt_src = df_evnt_src
        self.spark = SparkSession.builder.appName("Sequence Generation").getOrCreate()

    def calls_processing(self):
        df = self.df_call_src
        df = df.filter((col("DETL_PROD") == "Kerendia") & (col("BHID").isNotNull()))
        df = df.join(self.df_target, (df["BHID"] == self.df_target["BH_ID"]), "inner")
        PROMO_LAG_DAYS = 7
        start_date_boundary = pd.to_datetime(str(self.START_YRMO), format='%Y%m').date() - pd.Timedelta(days=PROMO_LAG_DAYS)
        end_date_boundary = (pd.to_datetime(str(self.END_YRMO), format='%Y%m') + pd.offsets.MonthEnd(0)).date() - pd.Timedelta(days=PROMO_LAG_DAYS)
        df = (df.withColumn("EXPOSED_ON", to_date(col("DATE_ID").cast("string"), "yyyyMMdd"))
              .filter(col("EXPOSED_ON").between(start_date_boundary, end_date_boundary)))
        df = df.withColumn("CONTENT_TAG", F.lit("General"))
        df = df.withColumnsRenamed({"BHID": "CUSTOMER_ID", "CHNNL_NM": "CHANNEL_ID", "DETL_PROD": "PRODUCT_ID", "CALL_ID": "INTERACTION_ID"})
        df = df.withColumn("ENGAGED_ON", col("EXPOSED_ON")).withColumn("ACTED_ON", col("EXPOSED_ON"))
        df = df.select("CUSTOMER_ID", "CHANNEL_ID", "CONTENT_TAG", "PRODUCT_ID", "INTERACTION_ID", "EXPOSED_ON", "ENGAGED_ON", "ACTED_ON")
        return df.toPandas()

    def email_processing(self):
        df = self.df_npp_src
        df = df.filter((col("PROD_BRAND_NM") == "KERENDIA") & (col("BAYER_HCP_ID").isNotNull()))
        PROMO_LAG_DAYS = 7
        start_date_boundary = pd.to_datetime(str(self.START_YRMO), format='%Y%m').date() - pd.Timedelta(days=PROMO_LAG_DAYS)
        end_date_boundary = (pd.to_datetime(str(self.END_YRMO), format='%Y%m') + pd.offsets.MonthEnd(0)).date() - pd.Timedelta(days=PROMO_LAG_DAYS)
        df = df.join(self.df_target, (df["CUST_ID"] == self.df_target["BH_ID"]), "inner")
        print(df.count())
        df = (df.withColumn("DATE_ID_DT", to_date(col("DATE_ID").cast("string"), "yyyyMMdd"))
              .filter(col("DATE_ID_DT").between(start_date_boundary, end_date_boundary)))
        print(df.count())
        df = df.dropDuplicates(["UNIQ_NPP_ID", "DATE_ID", "CUST_ID", "VEND_NM", "CHNL_NM", "MTR_NM"])
        df = df.withColumn("DEVC_VEND_PROD", F.concat_ws("_", F.col("DEVC"), F.col("VEND_PROD")))
        cols = ['VEND_NM', 'CHNL_NM', 'MTR_NM', "DEVC_VEND_PROD", "CPGN_NM", "TACT_NM"]
        for c in cols:
            df = df.withColumn(f"{c}_copy", F.col(c))
        merge_cols = ['CHNL_NM', 'VEND_NM', 'LITCODE', 'CNTT_NM', 'EMAIL_SUBJ_LN', 'CRTV_NM', 'TACT_NM', 'CPGN_NM', 'PRMRY_KEY_MSG', 'CF_NAME', 'CF_INDICATION', 'RPT_CHNL', 'CHNL_TYP', 'TACT_TYP']
        missing_placeholder = "_"
        def clean_merge_columns_spark(df, cols, missing_placeholder="_"):
            missing_strings = ["", " ", "None", "none", "NaN", "nan", "NULL", "null", "Null", "N/A", "n/a", "NA"]
            for col_name in cols:
                df = df.withColumn(col_name, F.col(col_name).cast("string"))
                df = df.withColumn(col_name, F.trim(F.col(col_name)))
                df = df.withColumn(col_name, F.when(F.col(col_name).isNull() | F.col(col_name).isin(missing_strings), F.lit(missing_placeholder)).otherwise(F.col(col_name)))
                df = df.withColumn(col_name, F.regexp_replace(F.col(col_name), r"[^\w]+", "_"))
                df = df.withColumn(col_name, F.regexp_replace(F.col(col_name), r"_+", "_"))
                df = df.withColumn(col_name, F.regexp_replace(F.col(col_name), r"^_+|_+$", ""))
                df = df.withColumn(col_name, F.when(F.col(col_name).isNull() | (F.col(col_name) == ""), F.lit(missing_placeholder)).otherwise(F.col(col_name)))
            return df
        df = clean_merge_columns_spark(df, merge_cols, missing_placeholder)
        df_tagg = clean_merge_columns_spark(self.df_tagg, merge_cols, missing_placeholder)
        df_tagg = df_tagg.dropDuplicates(merge_cols)
        df = df.join(df_tagg, on=merge_cols, how="left")
        df = df.withColumn("Content Category", F.when(F.col("Content Category").isin("-", "Not Needed") | F.col("Content Category").isNull(), F.lit("General")).otherwise(F.col("Content Category")))
        for c in cols:
            df = df.withColumn(c, F.col(f"{c}_copy"))
        df = df.drop(*[f"{c}_copy" for c in cols])
        df = df.withColumnsRenamed({"Content Category": "CONTENT_TAG"})
        _tact = F.upper(F.col('TACT_NM'))
        _prefix = F.upper(F.regexp_extract(F.col('TACT_NM'), r'(?i)(?:KERENDIA|Kerendiafinerenone)_([A-Za-z]+)', 1))
        df = df.withColumn('CHNL_NM_NEW',
            F.when(F.col('VEND_NM') == 'DOXIMITY',
                F.when(_tact.contains('DOCNEWS'), F.lit('DocNews')).when(_tact.contains('DOCSPOT'), F.lit('DocSpot')).when(_tact.contains('DOCPOINT'), F.lit('DocPoint')).when(_tact.contains('FORMULARY'), F.lit('FormularyOne')).when(_tact.contains('ANNOUNCE'), F.lit('Announcement')).when(_tact.rlike(r'\bDN\b'), F.lit('DocNews')).when(_tact.rlike(r'\bF1\b'), F.lit('FormularyOne')).when(_tact.rlike(r'\bAN/F1\b'), F.lit('FormularyOne')).when(_tact.rlike(r'\bAN\b'), F.lit('Announcement')))
            .when(F.col('VEND_NM') == 'MEDSCAPE',
                F.when(_prefix == 'EA', F.lit('3rd Party Email')).when(_prefix == 'BA', F.lit('BrandAlert')).when(_prefix == 'BS', F.lit('BrandSpot')).when(_prefix == 'PP', F.lit('PulsePoint Banner')).when(_prefix == 'FFQ', F.lit('FFQ')).when(_prefix == 'RECAP', F.lit('RECAP')).when(_prefix == 'TLM', F.lit('BANNER')).when(_prefix == 'PPEHR', F.lit('EHR')).when(_prefix == 'FAP', F.lit('FAP')).when(_prefix == 'DDTRD', F.lit('DDTRD')))
            .when(F.col('VEND_NM') == 'PEERDIRECT',
                F.when(_tact.contains('MDFI - EMAIL'), F.lit('MDFI - Email')).when(_tact.contains('MDFI - WEB'), F.lit('MDFI - Web')).when(_tact.contains('MDFI - BANNER'), F.lit('MDFI - Banner')).when(_tact.contains('MD MINUTE - EMAIL'), F.lit('MD Minute - Email')).when(_tact.contains('MD MINUTE - WEB'), F.lit('MD Minute - Web')).when(_tact.contains('MD MINUTE - BANNER'), F.lit('MD Minute - Banner')).when(_tact.contains('MD SHUFFLE - BANNER'), F.lit('MD Shuffle - Banner')).when(_tact.contains('MD SHUFFLE - WEB'), F.lit('MD Shuffle - Web')).when(_tact.contains('PV COMPENDIUM'), F.lit('FFQ')))
            .when(F.col('VEND_NM') == 'SFMC', F.lit('HQ Email'))
            .when(F.col('VEND_NM') == 'PULSEPOINT', F.lit('Banner')))
        _target_vendors = ['DOXIMITY', 'MEDSCAPE', 'SFMC', 'PEERDIRECT', 'PULSEPOINT']
        flag_lookup = (self.df_flag_new.filter(F.upper(F.col('VEND_NM')).isin(_target_vendors)).filter(F.col('Exposure Flag').isNotNull() | F.col('Engagement Flag').isNotNull() | F.col('Acted Flag').isNotNull()).groupBy('VEND_NM', 'CHNL_NM_NEW', 'MTR_NM').agg(F.max('Exposure Flag').alias('Exposure Flag'), F.max('Engagement Flag').alias('Engagement Flag'), F.max('Acted Flag').alias('Acted Flag')))
        _join_cols = ['VEND_NM', 'CHNL_NM_NEW', 'MTR_NM']
        df = df.withColumns({f'{c}_JK': F.upper(F.trim(F.col(c))) for c in _join_cols})
        flag_lookup = flag_lookup.withColumns({f'{c}_JK': F.upper(F.trim(F.col(c))) for c in _join_cols})
        _jk = [f'{c}_JK' for c in _join_cols]
        df = df.join(flag_lookup.drop('VEND_NM', 'CHNL_NM_NEW', 'MTR_NM'), on=_jk, how='left')
        df = df.drop(*_jk)
        df = df.filter(F.col('CHNL_NM_NEW').isNotNull() & (F.col('Exposure Flag').isNotNull() | F.col('Engagement Flag').isNotNull() | F.col('Acted Flag').isNotNull()))
        df = df.na.drop(subset=["UNIQ_NPP_ID"])
        df = df.select(["CUST_ID", "VEND_NM", "CHNL_NM_NEW", 'CONTENT_TAG', "PROD_BRAND_NM", "UNIQ_NPP_ID", 'Exposure Flag', 'Engagement Flag', 'Acted Flag', "DATE_ID_DT"]).dropDuplicates()
        keys = ["CUST_ID", "VEND_NM", "CHNL_NM_NEW", "CONTENT_TAG", "PROD_BRAND_NM", "UNIQ_NPP_ID"]
        w = Window.partitionBy(*keys).orderBy("DATE_ID_DT")
        exposure_df = df.filter(F.col("Exposure Flag") == 1).withColumn("event_seq", F.row_number().over(w)).select(*keys, "event_seq", F.col("DATE_ID_DT").alias("EXPOSED_ON"))
        engagement_df = df.filter(F.col("Engagement Flag") == 1).withColumn("event_seq", F.row_number().over(w)).select(*keys, "event_seq", F.col("DATE_ID_DT").alias("ENGAGED_ON"))
        acted_df = df.filter(F.col("Acted Flag") == 1).withColumn("event_seq", F.row_number().over(w)).select(*keys, "event_seq", F.col("DATE_ID_DT").alias("ACTED_ON"))
        result_df = exposure_df.join(engagement_df, keys + ["event_seq"], "full_outer").join(acted_df, keys + ["event_seq"], "full_outer").orderBy(*keys, "event_seq")
        result_df = result_df.withColumn("CHANNEL_ID", F.concat_ws("_", F.col("VEND_NM"), F.col("CHNL_NM_NEW")))
        result_df = result_df.withColumnsRenamed({"CUST_ID": "CUSTOMER_ID", "PROD_BRAND_NM": "PRODUCT_ID", "CONTENT_TAG": "CONTENT_TAG", "UNIQ_NPP_ID": "INTERACTION_ID"})
        result_df = result_df.select("CUSTOMER_ID", "CHANNEL_ID", "CONTENT_TAG", "PRODUCT_ID", "INTERACTION_ID", "EXPOSED_ON", "ENGAGED_ON", "ACTED_ON")
        result_df = result_df.dropDuplicates(["CUSTOMER_ID", "CHANNEL_ID", "CONTENT_TAG", "INTERACTION_ID"])
        return result_df.toPandas()

    def pp_processing(self):
        df = self.df_evnt_src
        df = df.filter((col("PROD_BRAND_NM") == "KERENDIA") & (col("CUST_HCP_ID").isNotNull()))
        df = df.join(self.df_target, (df["CUST_HCP_ID"] == self.df_target["BH_ID"]), "inner")
        df = df.dropDuplicates(["EVNT_ID", "DATE_ID", "CUST_HCP_ID", "EVNT_CAT", "EVNT_TYP", "EVNT_SUB_TYP", "ATND"])
        df = df.withColumnsRenamed({"CUST_HCP_ID": "CUSTOMER_ID", "EVNT_TYP": "CHANNEL_ID", "PROD_BRAND_NM": "PRODUCT_ID", "EVNT_ID": "INTERACTION_ID"})
        df = df.withColumn("CONTENT_TAG", F.lit("General"))
        PROMO_LAG_DAYS = 7
        start_date_boundary = pd.to_datetime(str(self.START_YRMO), format='%Y%m').date() - pd.Timedelta(days=PROMO_LAG_DAYS)
        end_date_boundary = (pd.to_datetime(str(self.END_YRMO), format='%Y%m') + pd.offsets.MonthEnd(0)).date() - pd.Timedelta(days=PROMO_LAG_DAYS)
        df = (df.withColumn("EXPOSED_ON", to_date(col("DATE_ID").cast("string"), "yyyyMMdd")).filter(col("EXPOSED_ON").between(start_date_boundary, end_date_boundary)))
        df = df.withColumn("ENGAGED_ON", col("EXPOSED_ON")).withColumn("ACTED_ON", col("EXPOSED_ON"))
        df = df.select("CUSTOMER_ID", "CHANNEL_ID", "CONTENT_TAG", "PRODUCT_ID", "INTERACTION_ID", "EXPOSED_ON", "ENGAGED_ON", "ACTED_ON")
        return df.toPandas()

    def get_transaction_idc(self):
        PROMO_LAG_DAYS = 7
        calls_idc = self.calls_processing()
        emails_idc = self.email_processing()
        events_idc = self.pp_processing()
        proto_idc = pd.concat([calls_idc, emails_idc, events_idc])
        for column in ['EXPOSED_ON', 'ENGAGED_ON', 'ACTED_ON']:
            proto_idc.loc[:, column] = proto_idc[column].astype(str)
            proto_idc.loc[:, 'RAW_' + column] = proto_idc[column]
            proto_idc.loc[:, column] = pd.to_datetime(proto_idc[column], errors='coerce').dt.date
        start_date_boundary = pd.to_datetime(str(self.START_YRMO), format='%Y%m').date() - pd.Timedelta(days=PROMO_LAG_DAYS)
        end_date_boundary = (pd.to_datetime(str(self.END_YRMO), format='%Y%m') + pd.offsets.MonthEnd(0)).date() - pd.Timedelta(days=PROMO_LAG_DAYS)
        proto_idc = proto_idc[(proto_idc['EXPOSED_ON'] >= start_date_boundary) & (proto_idc['EXPOSED_ON'] <= end_date_boundary)].copy(deep=True)
        for column in ['EXPOSED_ON', 'ENGAGED_ON', 'ACTED_ON']:
            proto_idc[column] = (pd.to_datetime(proto_idc[column]) + pd.Timedelta(days=PROMO_LAG_DAYS)).dt.date
        proto_idc['YRMO'] = pd.to_datetime(proto_idc['EXPOSED_ON']).dt.strftime("%Y%m")
        proto_idc['YRMO'] = pd.to_numeric(proto_idc['YRMO'])
        proto_idc = proto_idc[(proto_idc.YRMO >= self.START_YRMO) & (proto_idc.YRMO <= self.END_YRMO)].copy(deep=True)
        proto_idc = proto_idc[['CUSTOMER_ID', 'PRODUCT_ID', 'CHANNEL_ID', 'CONTENT_TAG', 'EXPOSED_ON', 'ACTED_ON', 'ENGAGED_ON', 'INTERACTION_ID', 'YRMO']].drop_duplicates()
        proto_idc['COUNT'] = 1
        proto_idc = standardize_columns(proto_idc, ['CHANNEL_ID', 'CONTENT_TAG'])
        rules = {'drop': ["CUSTOMER_ID", "EXPOSED_ON", "CHANNEL_ID"], 'fill': {1: ['COUNT'], 'NO_ID': ['INTERACTION_ID'], 'GENERAL': ['CONTENT_TAG'], pd.to_datetime(0): ['ACTED_ON', 'ENGAGED_ON']}}
        proto_idc = nan_handler(proto_idc, rules=rules)
        proto_idc = convert_to_numeric(proto_idc, 'COUNT')
        proto_idc = standardize_columns(proto_idc, ['CHANNEL_ID'])
        proto_idc['CONTENT_TAG'] = proto_idc['CONTENT_TAG'].replace(['NONE', None, np.nan, 'NAN', '', ' ', 'OTHERS'], 'GENERAL')
        proto_idc = proto_idc[(proto_idc.YRMO >= self.START_YRMO) & (proto_idc.YRMO <= self.END_YRMO)].copy(deep=True)
        print(f"Idc3 file shape:{proto_idc.shape}")
        proto_idc = proto_idc.copy()
        for c in ['CUSTOMER_ID', 'INTERACTION_ID', 'CHANNEL_ID', 'CONTENT_TAG']:
            proto_idc[c] = proto_idc[c].astype(str)
        for c in ['EXPOSED_ON', 'ACTED_ON', 'ENGAGED_ON']:
            proto_idc[c] = pd.to_datetime(proto_idc[c], errors='coerce')
        sort_cols = ['CUSTOMER_ID', 'INTERACTION_ID', 'CHANNEL_ID', 'CONTENT_TAG', 'EXPOSED_ON', 'ACTED_ON', 'ENGAGED_ON']
        proto_idc = proto_idc.sort_values(sort_cols, ascending=True).groupby(['CUSTOMER_ID', 'INTERACTION_ID', 'CHANNEL_ID', 'EXPOSED_ON'], as_index=False).head(1)
        proto_idc = proto_idc[['CUSTOMER_ID', 'PRODUCT_ID', 'CHANNEL_ID', 'CONTENT_TAG', 'EXPOSED_ON', 'ACTED_ON', 'ENGAGED_ON', 'INTERACTION_ID', 'YRMO', 'COUNT']]
        return proto_idc

    def process_counts(self, prod_idc, metric_type, channel_col='CHANNEL_ID'):
        channels = self.metric_mapping[metric_type].keys()
        idc_counts = prod_idc[prod_idc[channel_col].isin(channels)]
        idc_counts = idc_counts[(idc_counts.YRMO >= self.START_YRMO) & (idc_counts.YRMO <= self.END_YRMO)]
        date_col_map = {'R': 'EXPOSED_ON', 'I': 'ENGAGED_ON', 'E': 'ACTED_ON'}
        dc = date_col_map[metric_type]
        idc_counts[dc] = pd.to_datetime(idc_counts[dc], errors='coerce')
        idc_counts['YRMO'] = idc_counts[dc].dt.strftime('%Y%m')
        a = idc_counts[['CUSTOMER_ID', channel_col, 'YRMO', 'COUNT']].groupby(['CUSTOMER_ID', 'YRMO', channel_col]).COUNT.sum().to_frame('COUNT').reset_index()
        eng_counts = pd.pivot_table(a, values='COUNT', index=['CUSTOMER_ID', 'YRMO'], columns=channel_col, aggfunc=sum, fill_value=0)
        eng_counts.columns = [ch + '_' + self.metric_mapping[metric_type][ch] + '_' + metric_type for ch in eng_counts.columns.to_list()]
        return eng_counts.reset_index()

    def content_counts(self, transactions):
        idc_counts = transactions.copy()
        idc_counts = idc_counts[(idc_counts.YRMO >= self.START_YRMO) & (idc_counts.YRMO <= self.END_YRMO)]
        idc_counts['EXPOSED_ON'] = pd.to_datetime(idc_counts['EXPOSED_ON'], errors='coerce')
        idc_counts['YRMO'] = idc_counts.EXPOSED_ON.dt.strftime('%Y%m')
        a = idc_counts[['CUSTOMER_ID', 'CONTENT_TAG', 'YRMO', 'COUNT']].groupby(['CUSTOMER_ID', 'YRMO', 'CONTENT_TAG']).COUNT.sum().to_frame('COUNT').reset_index()
        eng_counts = pd.pivot_table(a, values='COUNT', index=['CUSTOMER_ID', 'YRMO'], columns='CONTENT_TAG', aggfunc=sum, fill_value=0)
        eng_counts.columns = [ch + '_COUNT' for ch in eng_counts.columns.to_list()]
        return eng_counts.reset_index()

    def get_counts(self, transactions=None):
        if transactions is None:
            transactions = self.get_transaction_idc()
        transactions['CONTENT_TAG'] = transactions['CONTENT_TAG'].replace(['NONE', None, np.nan, 'NAN', ''], 'GENERAL')
        count_dfs = [self.process_counts(transactions, 'R'), self.process_counts(transactions, 'I'), self.process_counts(transactions, 'E'), self.content_counts(transactions)]
        cnts = reduce(lambda x, y: pd.merge(x, y, on=['CUSTOMER_ID', 'YRMO'], how='outer', validate='one_to_one'), count_dfs).fillna(0)
        cnts.YRMO = cnts.YRMO.astype('Int64')
        return cnts

# =========================================================================
# SequenceProcessor class (inlined from data_processing.py)
# =========================================================================
class SequenceProcessor:
    def __init__(self):
        self.channel_encodings = None
        self.content_encodings = None
        self.channel_encodings_dict = {}
        self.content_encodings_dict = {}

    def preprocess_activity_data(self, idc, content_rename=None):
        mask = idc["CHANNEL_ID"].str.contains("BANNER", case=False, na=False)
        idc.loc[mask, "EXPOSED_ON"] = idc.loc[mask, "ENGAGED_ON"]
        idc = idc[idc["EXPOSED_ON"].notna()]
        activity_data = idc.astype({'CUSTOMER_ID': str, 'YRMO': int}).rename(columns={'CONTENT_TAG': 'CONTENT_CATEGORY'})
        activity_data = activity_data[['CUSTOMER_ID', 'EXPOSED_ON', 'CHANNEL_ID', 'CONTENT_CATEGORY', 'COUNT', 'YRMO', 'PRODUCT_ID', 'INTERACTION_ID']].copy()
        activity_data.loc[:, 'CONTENT_CATEGORY'] = activity_data['CONTENT_CATEGORY'].str.upper()
        activity_data['CHANNEL_PRODUCT'] = activity_data['CHANNEL_ID'].astype(str) + '&' + activity_data['PRODUCT_ID'].astype(str)
        activity_data = activity_data.drop('CHANNEL_ID', axis=1)
        return activity_data

    def create_encodings(self, activity_data):
        channel_list = sorted(list(activity_data['CHANNEL_PRODUCT'].unique()))
        total_contents = sorted(activity_data['CONTENT_CATEGORY'].unique())
        self.channel_encodings = pd.DataFrame({'CHANNEL_ID': channel_list + ['ZG', 'LG', 'MG', 'HG'], 'CHANNEL_CODE': range(1, len(channel_list) + 5)})
        self.content_encodings = pd.DataFrame({'CONTENT_CATEGORY': total_contents + ['ZG', 'LG', 'MG', 'HG'], 'CONTENT_CODE': range(1, len(total_contents) + 5)})
        self.channel_encodings_dict = dict(zip(self.channel_encodings['CHANNEL_ID'].str.upper(), self.channel_encodings['CHANNEL_CODE']))
        self.content_encodings_dict = dict(zip(self.content_encodings['CONTENT_CATEGORY'], self.content_encodings['CONTENT_CODE']))

    @staticmethod
    def convert_seq(seq):
        arr = np.array(seq)
        non_gap_indices = np.nonzero(arr != "GAP")[0]
        if len(non_gap_indices) == 0:
            return []
        gap_counts = np.zeros_like(non_gap_indices)
        if non_gap_indices[0] > 0:
            gap_counts[0] = non_gap_indices[0]
        for i in range(1, len(non_gap_indices)):
            gap_counts[i] = non_gap_indices[i] - non_gap_indices[i - 1] - 1
        conditions = [(gap_counts >= 0) & (gap_counts <= 3), (gap_counts >= 4) & (gap_counts <= 10), (gap_counts >= 11) & (gap_counts <= 17), (gap_counts > 17)]
        choices = ["ZG", "LG", "MG", "HG"]
        gap_types = np.select(conditions, choices, default="ZG")
        seq_out = []
        for i in range(len(non_gap_indices)):
            seq_out.append(gap_types[i])
            seq_out.append(arr[non_gap_indices[i]])
        return seq_out

    def generate_gap_sequences(self, df_final):
        gap_grp = pd.DataFrame()
        channel_seqs = df_final['full_channelName_seq'].apply(self.convert_seq)
        gap_grp['full_channelName_seq_GAP_grp'] = channel_seqs
        gap_grp['full_channelCode_seq_GAP_grp'] = channel_seqs.apply(lambda x: [self.channel_encodings_dict[i] for i in x])
        content_seqs = df_final['full_contentName_seq'].apply(self.convert_seq)
        gap_grp['full_contentName_seq_GAP_grp'] = content_seqs
        gap_grp['full_contentCode_seq_GAP_grp'] = content_seqs.apply(lambda x: [self.content_encodings_dict[i] for i in x])
        return gap_grp

    def process_sequences(self, idc, seq_gen_obj, start_date, end_date, content_rename=None):
        activity_data = self.preprocess_activity_data(idc, content_rename)
        self.create_encodings(activity_data)
        activity_data = pd.merge(activity_data, self.channel_encodings, left_on='CHANNEL_PRODUCT', right_on='CHANNEL_ID', how='left', validate='many_to_many').merge(self.content_encodings, on='CONTENT_CATEGORY', how='left', validate='many_to_many')
        activity_data = activity_data[['CUSTOMER_ID', 'PRODUCT_ID', 'CHANNEL_PRODUCT', 'CHANNEL_CODE', 'CONTENT_CATEGORY', 'CONTENT_CODE', 'COUNT', 'EXPOSED_ON', 'YRMO']]
        p3 = list(activity_data.YRMO.nsmallest(3))
        c3 = list(activity_data.YRMO.nlargest(3))
        activity_data['TimeFrame'] = np.select([activity_data['YRMO'].isin(p3), activity_data['YRMO'].isin(c3)], ['P3', 'C3'], default='0')
        activity_data['EXPOSED_ON'] = pd.to_datetime(activity_data.EXPOSED_ON)
        sequence_data = seq_gen_obj.journey_rollup(activity_data=activity_data, start_date=start_date, end_date=end_date)
        sequence_data.columns = ['CUSTOMER_ID', 'full_channelName_seq', 'full_channelCode_seq', 'channelName_seq', 'channelCode_seq', 'full_contentName_seq', 'full_contentCode_seq', 'contentName_seq', 'contentCode_seq', 'timeSeq']
        channel_list = list(activity_data['CHANNEL_PRODUCT'].unique())
        content_list = list(activity_data['CONTENT_CATEGORY'].unique())
        channel_counts = seq_gen_obj.channel_count(master_data=sequence_data, event_name="channelName_seq", channel_list=channel_list)
        content_counts = seq_gen_obj.content_count(master_data=sequence_data, event_name="channelName_seq", content_list=content_list)
        final_df = reduce(lambda left, right: pd.merge(left, right, on=['CUSTOMER_ID'], how='inner', validate='one_to_one'), [sequence_data, channel_counts, content_counts])
        gap_sequences = self.generate_gap_sequences(sequence_data)
        return pd.concat([final_df, gap_sequences], axis=1)

print("Transactions and SequenceProcessor classes loaded (no external file dependency).")

In [0]:
import pandas as pd
import numpy as np
from functools import reduce
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql.functions import col, to_date, when, lit, upper
from pyspark.sql.types import StringType, NumericType
from pyspark.sql.window import Window
import re

# -- Spark session --
spark = SparkSession.builder.appName("Email_Downstream_EDA").getOrCreate()

# -- Config: adjust these to match your run --
START_YRMO = 202406          # p3_start_yrmo from generate_mdb
END_YRMO   = 202506          # absolute_end_yrmo from generate_mdb
PROMO_LAG_DAYS = 7
P3_LAG_MONTH = 1

# -- Flat files --
df_flag_new = pd.read_excel(
    "/Workspace/Users/wenjie.chen@bayer.com/OE/GA Run/Data_for_production/"
    "5_5_2026_Bayer_OE_updated_channel_eng_exp_flag_latest.xlsx"
)
df_flag_new = spark.createDataFrame(df_flag_new)

df_tagg = pd.read_excel(
    "/Workspace/Users/wenjie.chen@bayer.com/OE/GA Run/Data_for_production/"
    "Content_tagging_6_5.xlsx"
)
df_tagg['LITCODE'] = df_tagg['LITCODE'].astype(str)
df_tagg = spark.createDataFrame(df_tagg)

# -- Target universe --
path_target = "/Volumes/ph-com-ai-us-prod-catalog-2553213839599380/oe_journey/oe_volume_space/notebook_output/13_5_oe_modelling_universe_final"
df_target = spark.read.format("delta").load(path_target)
df_target = df_target.filter(col("modelling_flag") == "Yes")

# -- Metric mapping (from config Excel) --
file_path = "/Workspace/Users/wenjie.chen@bayer.com/OE/GA Run/Data/mdb_prep_config.xlsx"
xls = pd.ExcelFile(file_path)
df_metric_mapping = pd.read_excel(xls, sheet_name="Metric Mappings")
metric_mapping = {}
for _, row in df_metric_mapping.iterrows():
    cat = row["Category"]
    evt = row["Event Type"]
    mapped = row["Mapped Metric"]
    if cat not in metric_mapping:
        metric_mapping[cat] = {}
    metric_mapping[cat][evt] = mapped

print("Setup complete.")
print(f"START_YRMO={START_YRMO}, END_YRMO={END_YRMO}")
print(f"df_target rows: {df_target.count()}")
print(f"metric_mapping keys: {list(metric_mapping.keys())}")

In [0]:
# =========================================================================
# Promotional cohorts (same source as DX_Patient_Features' CohortGenerator, family = "promotional")
# File is parquet (not csv); columns: cohort_id, prediction_month, dataset, offset_days,
# lookback_start_date, lookback_end_date. 28 cohorts c1..c28, offset 17 days.
# NBA 3.0 "I01" (inference) == latest cohort here.
# =========================================================================
COHORT_PATH = ("/Volumes/ph-com-ai-us-prod-catalog-2553213839599380/kerendia_nba_3/"
               "kerendia_nba_3_1_volume/cohorts/promotional/cohorts.parquet/")

cohort_anchor_date = (spark.read.parquet(COHORT_PATH)
                      .withColumn("COHORT_NUM", F.regexp_extract("cohort_id", r"(\d+)", 1).cast("int"))
                      .orderBy("COHORT_NUM"))

# Pick the cohort to process (default: latest cohort, equivalent of NBA 3.0 I01)
COHORT_ID = cohort_anchor_date.orderBy(F.col("COHORT_NUM").desc()).first()["cohort_id"]

sel = cohort_anchor_date.filter(F.col("cohort_id") == COHORT_ID).first()
assert sel is not None, f"Cohort {COHORT_ID} not found"
LOOKBACK_START_DATE = sel["lookback_start_date"]
LOOKBACK_END_DATE   = sel["lookback_end_date"]
LOOKBACK_START_DATE_STR = LOOKBACK_START_DATE.strftime("%Y%m%d")

# Drive the downstream YRMO window from the UNION of all cohort lookbacks (replaces the hard-coded values above),
# so proto_idc (Step 4) covers every cohort and features can be built at HCP x cohort level.
_rng = cohort_anchor_date.agg(F.min("lookback_start_date").alias("s"), F.max("lookback_end_date").alias("e")).first()
START_YRMO = int(_rng["s"].strftime("%Y%m"))
END_YRMO   = int(_rng["e"].strftime("%Y%m"))

print(f"Cohorts loaded: {cohort_anchor_date.count()} | selected {COHORT_ID} ({sel['dataset']}, prediction month {sel['prediction_month']})")
print(f"LOOKBACK_START_DATE={LOOKBACK_START_DATE} ({LOOKBACK_START_DATE_STR}), LOOKBACK_END_DATE={LOOKBACK_END_DATE}")
print(f"START_YRMO={START_YRMO}, END_YRMO={END_YRMO}")
cohort_anchor_date.drop("COHORT_NUM").display()

In [0]:
# ===== STEP 1: Load the raw NPP (Non-Personal Promotion) source table =====
# This is the primary input to email_processing()

df_raw_npp = df_npp_src

print(f"Raw NPP table row count: {df_raw_npp.count():,}")
print(f"Raw NPP table column count: {len(df_raw_npp.columns)}")
print("\nSchema:")
df_raw_npp.printSchema()

print("\nSample rows (KERENDIA only):")
df_raw_npp.filter(
    (col("PROD_BRAND_NM") == "KERENDIA") & col("BAYER_HCP_ID").isNotNull()
).select(
    "CUST_ID", "VEND_NM", "CHNL_NM", "MTR_NM", "TACT_NM",
    "PROD_BRAND_NM", "DATE_ID", "UNIQ_NPP_ID"
).show(5, truncate=False)

In [0]:
# ===== STEP 2: Reproduce email_processing() and capture the output =====
# This is the DataFrame returned by Transactions.email_processing()

generator = Transactions(
    start_yrmo=START_YRMO,
    end_yrmo=END_YRMO,
    product_key='KERENDIA',
    country='US',
    metric_mapping=metric_mapping,
    df_flag_new=df_flag_new,
    df_target=df_target,
    df_tagg=df_tagg,
    df_call_src=df_call_src,
    df_npp_src=df_npp_src,
    df_evnt_src=df_evnt_src,
    env='dev',
)

emails_idc = generator.email_processing()

print(f"email_processing() output shape: {emails_idc.shape}")
print(f"Columns: {emails_idc.columns.tolist()}")
print(f"\nUnique channels ({emails_idc['CHANNEL_ID'].nunique()}):")
print(emails_idc['CHANNEL_ID'].value_counts())
print(f"\nUnique content tags ({emails_idc['CONTENT_TAG'].nunique()}):")
print(emails_idc['CONTENT_TAG'].value_counts())
exposed = pd.to_datetime(emails_idc['EXPOSED_ON'], errors='coerce')
print(f"\nDate range: {exposed.min()} to {exposed.max()}")
print(f"Unique customers: {emails_idc['CUSTOMER_ID'].nunique():,}")
emails_idc.head(10)

In [0]:
# ===== STEP 3: Concatenation in get_transaction_idc() =====
# email_processing() output is concat'd with calls_processing() and pp_processing()

calls_idc = generator.calls_processing()
events_idc = generator.pp_processing()

proto_idc = pd.concat([calls_idc, emails_idc, events_idc])

print(f"calls_idc shape:  {calls_idc.shape}")
print(f"emails_idc shape: {emails_idc.shape}")
print(f"events_idc shape: {events_idc.shape}")
print(f"proto_idc shape (after concat): {proto_idc.shape}")

# Tag each row's source for traceability
proto_idc_tagged = proto_idc.copy()
proto_idc_tagged['_SOURCE'] = (
    ['calls'] * len(calls_idc) +
    ['emails'] * len(emails_idc) +
    ['events'] * len(events_idc)
)
print(f"\nRow counts by source:")
print(proto_idc_tagged['_SOURCE'].value_counts())

print(f"\nEmail rows — channel distribution after concat:")
print(proto_idc_tagged[proto_idc_tagged['_SOURCE'] == 'emails']['CHANNEL_ID'].value_counts().head(15))

In [0]:
# ===== STEP 4: Full get_transaction_idc() post-processing =====
# Date string conversion, +7 day lag, YRMO filter, standardize, nan_handler, dedup

# 4a. Date conversion
for column in ['EXPOSED_ON', 'ENGAGED_ON', 'ACTED_ON']:
    proto_idc.loc[:, column] = proto_idc[column].astype(str)
    proto_idc.loc[:, 'RAW_' + column] = proto_idc[column]
    proto_idc.loc[:, column] = pd.to_datetime(proto_idc[column], errors='coerce').dt.date

# 4b. Filter by date boundaries
start_date_boundary = pd.to_datetime(str(START_YRMO), format='%Y%m').date() - pd.Timedelta(days=PROMO_LAG_DAYS)
end_date_boundary = (pd.to_datetime(str(END_YRMO), format='%Y%m') + pd.offsets.MonthEnd(0)).date() - pd.Timedelta(days=PROMO_LAG_DAYS)
print(f"Date boundaries: {start_date_boundary} to {end_date_boundary}")

before_filter = len(proto_idc)
proto_idc = proto_idc[
    (proto_idc['EXPOSED_ON'] >= start_date_boundary) &
    (proto_idc['EXPOSED_ON'] <= end_date_boundary)
].copy(deep=True)
print(f"Rows before date filter: {before_filter:,} → after: {len(proto_idc):,}")

# 4c. Apply +7 day promotional lag
for column in ['EXPOSED_ON', 'ENGAGED_ON', 'ACTED_ON']:
    proto_idc[column] = (
        pd.to_datetime(proto_idc[column]) + pd.Timedelta(days=PROMO_LAG_DAYS)
    ).dt.date

# 4d. YRMO assignment and filter
proto_idc['YRMO'] = pd.to_datetime(proto_idc['EXPOSED_ON']).dt.strftime('%Y%m')
proto_idc['YRMO'] = pd.to_numeric(proto_idc['YRMO'])
proto_idc = proto_idc[
    (proto_idc.YRMO >= START_YRMO) & (proto_idc.YRMO <= END_YRMO)
].copy(deep=True)
print(f"Rows after YRMO filter: {len(proto_idc):,}")

# 4e. Select columns, dedup, add COUNT
proto_idc = proto_idc[[
    'CUSTOMER_ID', 'PRODUCT_ID', 'CHANNEL_ID', 'CONTENT_TAG',
    'EXPOSED_ON', 'ACTED_ON', 'ENGAGED_ON', 'INTERACTION_ID', 'YRMO'
]].drop_duplicates()
proto_idc['COUNT'] = 1

# 4f. Standardize, nan_handler, dedup
proto_idc = standardize_columns(proto_idc, ['CHANNEL_ID', 'CONTENT_TAG'])
rules = {
    'drop': ['CUSTOMER_ID', 'EXPOSED_ON', 'CHANNEL_ID'],
    'fill': {
        1: ['COUNT'],
        'NO_ID': ['INTERACTION_ID'],
        'GENERAL': ['CONTENT_TAG'],
        pd.to_datetime(0): ['ACTED_ON', 'ENGAGED_ON']
    }
}
proto_idc = nan_handler(proto_idc, rules=rules)
proto_idc = convert_to_numeric(proto_idc, 'COUNT')
proto_idc = standardize_columns(proto_idc, ['CHANNEL_ID'])
proto_idc['CONTENT_TAG'] = proto_idc['CONTENT_TAG'].replace(
    ['NONE', None, np.nan, 'NAN', '', ' ', 'OTHERS'], 'GENERAL'
)
proto_idc = proto_idc[
    (proto_idc.YRMO >= START_YRMO) & (proto_idc.YRMO <= END_YRMO)
].copy(deep=True)

# 4g. Type casting + final dedup
text_cols = ['CUSTOMER_ID', 'INTERACTION_ID', 'CHANNEL_ID', 'CONTENT_TAG']
for c in text_cols:
    proto_idc[c] = proto_idc[c].astype(str)
date_cols = ['EXPOSED_ON', 'ACTED_ON', 'ENGAGED_ON']
for c in date_cols:
    proto_idc[c] = pd.to_datetime(proto_idc[c], errors='coerce')

sort_cols = [
    'CUSTOMER_ID', 'INTERACTION_ID', 'CHANNEL_ID', 'CONTENT_TAG',
    'EXPOSED_ON', 'ACTED_ON', 'ENGAGED_ON'
]
proto_idc = (
    proto_idc
    .sort_values(sort_cols, ascending=True)
    .groupby(['CUSTOMER_ID', 'INTERACTION_ID', 'CHANNEL_ID', 'EXPOSED_ON'], as_index=False)
    .head(1)
)
proto_idc = proto_idc[[
    'CUSTOMER_ID', 'PRODUCT_ID', 'CHANNEL_ID', 'CONTENT_TAG',
    'EXPOSED_ON', 'ACTED_ON', 'ENGAGED_ON', 'INTERACTION_ID',
    'YRMO', 'COUNT'
]]

print(f"\nFinal get_transaction_idc() shape: {proto_idc.shape}")
print(f"Unique channels: {proto_idc['CHANNEL_ID'].nunique()}")
print(proto_idc['CHANNEL_ID'].value_counts().head(15))
print(f"\nYRMO distribution:")
print(proto_idc['YRMO'].value_counts().sort_index())
proto_idc.head(10)

In [0]:
# ===== STEP 5: get_counts() — R/I/E metric pivots + content counts =====
# This pivots proto_idc into channel × metric × YRMO counts

transactions = proto_idc.copy()
transactions['CONTENT_TAG'] = transactions['CONTENT_TAG'].replace(
    ['NONE', None, np.nan, 'NAN', ''], 'GENERAL'
)

def process_counts_local(prod_idc, metric_type, channel_col='CHANNEL_ID'):
    channels = metric_mapping[metric_type].keys()
    idc_counts = prod_idc[prod_idc[channel_col].isin(channels)].copy()
    idc_counts = idc_counts[(idc_counts.YRMO >= START_YRMO) & (idc_counts.YRMO <= END_YRMO)]
    date_col_map = {'R': 'EXPOSED_ON', 'I': 'ENGAGED_ON', 'E': 'ACTED_ON'}
    dc = date_col_map[metric_type]
    idc_counts[dc] = pd.to_datetime(idc_counts[dc], errors='coerce')
    idc_counts['YRMO'] = idc_counts[dc].dt.strftime('%Y%m')
    a = idc_counts[['CUSTOMER_ID', channel_col, 'YRMO', 'COUNT']].groupby(
        ['CUSTOMER_ID', 'YRMO', channel_col]
    ).COUNT.sum().to_frame('COUNT').reset_index()
    eng_counts = pd.pivot_table(a, values='COUNT', index=['CUSTOMER_ID', 'YRMO'],
                                columns=channel_col, aggfunc=sum, fill_value=0)
    eng_counts.columns = [
        ch + '_' + metric_mapping[metric_type][ch] + '_' + metric_type
        for ch in eng_counts.columns.tolist()
    ]
    return eng_counts.reset_index()

def content_counts_local(trans):
    idc_counts = trans.copy()
    idc_counts = idc_counts[(idc_counts.YRMO >= START_YRMO) & (idc_counts.YRMO <= END_YRMO)]
    idc_counts['EXPOSED_ON'] = pd.to_datetime(idc_counts['EXPOSED_ON'], errors='coerce')
    idc_counts['YRMO'] = idc_counts.EXPOSED_ON.dt.strftime('%Y%m')
    a = idc_counts[['CUSTOMER_ID', 'CONTENT_TAG', 'YRMO', 'COUNT']].groupby(
        ['CUSTOMER_ID', 'YRMO', 'CONTENT_TAG']
    ).COUNT.sum().to_frame('COUNT').reset_index()
    eng_counts = pd.pivot_table(a, values='COUNT', index=['CUSTOMER_ID', 'YRMO'],
                                columns='CONTENT_TAG', aggfunc=sum, fill_value=0)
    eng_counts.columns = [ch + '_COUNT' for ch in eng_counts.columns.tolist()]
    return eng_counts.reset_index()

count_R = process_counts_local(transactions, 'R')
count_I = process_counts_local(transactions, 'I')
count_E = process_counts_local(transactions, 'E')
count_content = content_counts_local(transactions)

cnts = reduce(
    lambda x, y: pd.merge(x, y, on=['CUSTOMER_ID', 'YRMO'], how='outer', validate='one_to_one'),
    [count_R, count_I, count_E, count_content]
).fillna(0)
cnts['YRMO'] = cnts['YRMO'].astype('Int64')

print(f"R counts shape: {count_R.shape}, columns: {count_R.columns.tolist()[:5]}...")
print(f"I counts shape: {count_I.shape}, columns: {count_I.columns.tolist()[:5]}...")
print(f"E counts shape: {count_E.shape}, columns: {count_E.columns.tolist()[:5]}...")
print(f"Content counts shape: {count_content.shape}")
print(f"\nMerged counts shape: {cnts.shape}")
print(f"All count columns ({len(cnts.columns)}):")
for c in sorted(cnts.columns):
    print(f"  {c}")
cnts.head(5)

In [0]:
# ===== STEP 6: SequenceProcessor.process_sequences() =====
# Generates channel/content sequences, encodings, gap-group sequences, and counts

seq_gen = zs_pa_dataprep  # alias the inlined class

transactions_seq = transactions.copy()
transactions_seq['PRODUCT_ID'] = 'KERENDIA'

seq_gen_instance = seq_gen(
    customer_id='CUSTOMER_ID',
    channel_name_col='CHANNEL_PRODUCT',
    channel_number_col='CHANNEL_CODE',
    content_name_col='CONTENT_CATEGORY',
    content_number_col='CONTENT_CODE',
    date_column='EXPOSED_ON'
)

# Use a single date range for EDA (first instance from config)
df_date_ranges = pd.read_excel(xls, sheet_name="Date Ranges")
date_ranges = list(df_date_ranges.itertuples(index=False, name=None))
start_date, end_date = date_ranges[0]

processor = SequenceProcessor()
final_df = processor.process_sequences(
    idc=transactions_seq,
    seq_gen_obj=seq_gen_instance,
    start_date=start_date,
    end_date=end_date
)

print(f"SequenceProcessor output shape: {final_df.shape}")
print(f"Columns: {final_df.columns.tolist()}")
print(f"\nChannel encodings ({len(processor.channel_encodings)} entries):")
print(processor.channel_encodings.head(10))
print(f"\nContent encodings ({len(processor.content_encodings)} entries):")
print(processor.content_encodings)
print(f"\nSample sequences (first 3 rows):")
final_df[['CUSTOMER_ID', 'channelName_seq', 'contentName_seq', 'timeSeq']].head(3)

In [0]:
# ===== STEP 7: process_timeframes() — C1/C3/P3/P6/P12 window sums =====
# Aggregates counts into time windows and adds touchpoint rollup features

from datetime import datetime

def process_timeframes(counts_df, start_date, end_date):
    if isinstance(start_date, str):
        start_dt = datetime.strptime(start_date, '%m/%d/%Y')
    else:
        start_dt = start_date

    C1 = {int(start_dt.strftime('%Y%m'))}
    C3 = set()
    current_dt = start_dt
    for _ in range(3):
        C3.add(int(current_dt.strftime('%Y%m')))
        if current_dt.month == 12:
            current_dt = datetime(current_dt.year + 1, 1, 1)
        else:
            current_dt = datetime(current_dt.year, current_dt.month + 1, 1)
    P3 = set()
    current_dt = start_dt
    for _ in range(3):
        if current_dt.month == 1:
            current_dt = datetime(current_dt.year - 1, 12, 1)
        else:
            current_dt = datetime(current_dt.year, current_dt.month - 1, 1)
        P3.add(int(current_dt.strftime('%Y%m')))
    P6 = set()
    current_dt = start_dt
    for _ in range(6):
        if current_dt.month == 1:
            current_dt = datetime(current_dt.year - 1, 12, 1)
        else:
            current_dt = datetime(current_dt.year, current_dt.month - 1, 1)
        P6.add(int(current_dt.strftime('%Y%m')))
    P12 = set()
    current_dt = start_dt
    for _ in range(12):
        if current_dt.month == 1:
            current_dt = datetime(current_dt.year - 1, 12, 1)
        else:
            current_dt = datetime(current_dt.year, current_dt.month - 1, 1)
        P12.add(int(current_dt.strftime('%Y%m')))

    timeframes = {'C1': C1, 'C3': C3, 'P3': P3, 'P6': P6, 'P12': P12}
    df = counts_df.copy()
    result = pd.DataFrame(df['CUSTOMER_ID'].unique(), columns=['CUSTOMER_ID'])
    for label, timeframe in timeframes.items():
        mask = df['YRMO'].isin(timeframe)
        aggregated = df[mask].groupby('CUSTOMER_ID').sum()
        aggregated.drop(columns='YRMO', inplace=True)
        cols = aggregated.columns.tolist()
        exposure_cols = [
            c for c in cols if ('_R_' in c or c.endswith('_R')) and 'BANNER' not in c.upper()
        ] + [
            c for c in cols if ('_I_' in c or c.endswith('_I')) and 'BANNER' in c.upper()
        ]
        exposure_all_cols = [c for c in cols if '_R_' in c or c.endswith('_R')]
        engagement_cols = [c for c in cols if '_I_' in c or c.endswith('_I')]
        interaction_cols = [c for c in cols if '_E_' in c or c.endswith('_E')]
        aggregated['EXPOSURE_TOUCHPOINT'] = aggregated[exposure_all_cols].sum(axis=1) if exposure_all_cols else 0
        aggregated['EXPOSURE_TOUCHPOINT_BANNER_ENGAGEMENT'] = aggregated[exposure_cols].sum(axis=1) if exposure_cols else 0
        aggregated['ENGAGEMENT_TOUCHPOINT'] = aggregated[engagement_cols].sum(axis=1) if engagement_cols else 0
        aggregated['INTERACTION_TOUCHPOINT'] = aggregated[interaction_cols].sum(axis=1) if interaction_cols else 0
        aggregated.columns = [f"{c}_{label}" for c in aggregated.columns]
        result = result.merge(aggregated, on='CUSTOMER_ID', how='left', validate='one_to_one')
    result.fillna(0, inplace=True)
    return result, timeframes

result_df, calculated_timeframes = process_timeframes(cnts, start_date, end_date)

print(f"Timeframes computed:")
for k, v in calculated_timeframes.items():
    print(f"  {k}: {sorted(v)}")
print(f"\nResult shape: {result_df.shape}")
print(f"Sample columns (first 20):")
for c in result_df.columns[:20]:
    print(f"  {c}")
result_df.head(5)

In [0]:
# ===== STEP 8: Final MDB merge chain =====
# Merge: final_df (sequences) + result_df (timeframe counts) + demographics + CVI + P4/P6/P12 sales

# 8a. Merge sequences + timeframe counts
mdb = pd.merge(final_df, result_df, on='CUSTOMER_ID', how='left', validate='one_to_one').fillna(0)
print(f"After sequences + counts merge: {mdb.shape}")

# 8b. Merge demographics
path_demog = "/Volumes/ph-com-ai-us-prod-catalog-2553213839599380/oe_journey/oe_volume_space/notebook_output/temp_kerendia_hcp_attr"
hcp_demog = spark.read.format("delta").load(path_demog)
hcp_demog_joined = hcp_demog.join(df_target, on="BH_ID", how="inner")
hcp_demog_joined = hcp_demog_joined.select(
    "BH_ID", "HCP_SPECIALTY", "HCP_MEDICAL_DEGREE", "HCO_TYPE_DESCRIPTION",
    "MAJOR_CLASS_OF_TRADE", "HCO_SPECIALTY", "PROD_BRAND_NM", "TC_NM",
    "BRAND_SALES_SCORE", "COMPETITOR_SALES_SCORE", "CALLS_SCORE",
    "CLAIMS_SCORE", "SPECIALTY_SCORE", "CARESET_VALIDATION_FLAG",
    "CARESET_NEW_AFFL_FLAG", "PARENT_ACCOUNT_PRIORITY_SCORE",
    "PROXIMITY_SCORE", "RELATIONSHIP_STRENGTH_SCORE", "EMPLOYMENT_FLAG",
    "SOLICITED_FEEDBACK_FLAG", "FIELD_FEEDBACK_FLAG", "FIELD_FEEDBACK_SCORE",
    "FINAL_SCORE", "BRAND_RANK", "PRIMARY_AFFILIATION_CONFIDENCE_LEVEL"
)
string_cols = [f.name for f in hcp_demog_joined.schema.fields if isinstance(f.dataType, StringType)]
numeric_cols = [f.name for f in hcp_demog_joined.schema.fields if isinstance(f.dataType, NumericType)]
hcp_demog_joined = hcp_demog_joined.fillna("other", subset=string_cols).fillna(0, subset=numeric_cols)
demog = hcp_demog_joined.toPandas()
demog.rename(columns={'BH_ID': 'CUSTOMER_ID', 'PROD_BRAND_NM': 'PRODUCT_ID', 'HCP_SPECIALTY': 'SPECIALTY'}, inplace=True)
demog.drop_duplicates(['CUSTOMER_ID'], inplace=True)

mdb = pd.merge(mdb, demog, on='CUSTOMER_ID', how='left', validate='one_to_one')
print(f"After demographics merge: {mdb.shape}")

print(f"\n===== FINAL MDB (Instance 1) =====")
print(f"Shape: {mdb.shape}")
print(f"Total columns: {len(mdb.columns)}")
print(f"Unique customers: {mdb['CUSTOMER_ID'].nunique():,}")
print(f"\nColumn groups:")
seq_cols = [c for c in mdb.columns if 'seq' in c.lower()]
count_cols = [c for c in mdb.columns if c.endswith(('_C1', '_C3', '_P3', '_P6', '_P12'))]
demog_cols = [c for c in mdb.columns if c in demog.columns and c != 'CUSTOMER_ID']
print(f"  Sequence columns: {len(seq_cols)}")
print(f"  Timeframe count columns: {len(count_cols)}")
print(f"  Demographic columns: {len(demog_cols)}")
print(f"  Other columns: {len(mdb.columns) - len(seq_cols) - len(count_cols) - len(demog_cols) - 1}")

print(f"\nAll columns:")
for i, c in enumerate(mdb.columns):
    print(f"  {i+1:3d}. {c}")

mdb.head(3)

# Content Tagging Downstream Flow

## Source
`Content_tagging_6_5.xlsx` → loaded as `df_tagg` (Spark DataFrame, 15 columns).
The key output column is **`Content Category`**, which maps each NPP record to a content theme.

---

## Where `df_tagg` Is Used

### 1. `email_processing()` — the ONLY direct consumer
- NPP records are joined with `df_tagg` on **14 merge columns** (`CHNL_NM, VEND_NM, LITCODE, CNTT_NM, EMAIL_SUBJ_LN, CRTV_NM, TACT_NM, CPGN_NM, PRMRY_KEY_MSG, CF_NAME, CF_INDICATION, RPT_CHNL, CHNL_TYP, TACT_TYP`)
- `Content Category` is renamed to **`CONTENT_TAG`**; nulls / "-" / "Not Needed" are filled with `"General"`
- **Note**: `calls_processing()` and `pp_processing()` hard-code `CONTENT_TAG = "General"`, so content tagging affects **email/NPP data only**

### 2. `get_transaction_idc()` — standardization
- `standardize_columns(proto_idc, ['CONTENT_TAG'])` → uppercased, special chars → underscores
- NaN/blanks → `'GENERAL'`

### 3. `get_counts()` → `content_counts()` — per-tag pivots
- Pivots `CONTENT_TAG` into **`{TAG}_COUNT`** columns per `CUSTOMER_ID × YRMO`
- Example output columns: `GENERAL_COUNT`, `PATIENT_ADHERENCE_SUPPORT_COUNT`, `COST_COUNT`, `ACCESS_COVERAGE_COUNT`, `BASIC_PRODUCT_INFO_COUNT`, etc.

### 4. `SequenceProcessor.preprocess_activity_data()` — sequences
- `CONTENT_TAG` → renamed to `CONTENT_CATEGORY` (uppercased)
- Generates:
  - `full_contentName_seq` — full daily content sequence (with GAPs)
  - `full_contentCode_seq` — integer-encoded version
  - `contentName_seq` — activity-only content sequence
  - `contentCode_seq` — integer-encoded version
  - `full_contentName_seq_GAP_grp` — gap-grouped content sequence
  - `full_contentCode_seq_GAP_grp` — integer-encoded version
- `content_count()` from `zs_pa_dataprep` also generates per-content-tag count columns from sequences

### 5. `process_timeframes()` — time-window aggregations
- Each `{TAG}_COUNT` column is summed into **5 time windows**:
  - `{TAG}_COUNT_C1`, `{TAG}_COUNT_C3`, `{TAG}_COUNT_P3`, `{TAG}_COUNT_P6`, `{TAG}_COUNT_P12`
- For 16 content tags → **80 timeframe features** from content tagging alone

### 6. Final MDB
- All above features merge into the model training MDB

In [0]:
# ===== CONTENT TAGGING: df_tagg Lookup + All Downstream Features =====

print("="*70)
print("A. df_tagg LOOKUP TABLE (Content_tagging_6_5.xlsx)")
print("="*70)
df_tagg_pd = df_tagg.toPandas()
print(f"Shape: {df_tagg_pd.shape}")
print(f"Columns: {df_tagg_pd.columns.tolist()}")
print(f"\nUnique Content Category values ({df_tagg_pd['Content Category'].nunique()}):")
print(df_tagg_pd['Content Category'].value_counts(dropna=False))

print("\n" + "="*70)
print("B. CONTENT_TAG in email_processing() output (emails_idc)")
print("="*70)
print(f"emails_idc CONTENT_TAG distribution ({emails_idc['CONTENT_TAG'].nunique()} tags):")
print(emails_idc['CONTENT_TAG'].value_counts())
print(f"\n% rows with non-General content tag: {(emails_idc['CONTENT_TAG'] != 'General').mean()*100:.1f}%")

print("\n" + "="*70)
print("C. CONTENT_TAG after get_transaction_idc() (proto_idc — standardized)")
print("="*70)
print(f"proto_idc CONTENT_TAG distribution ({proto_idc['CONTENT_TAG'].nunique()} tags):")
print(proto_idc['CONTENT_TAG'].value_counts())

print("\n" + "="*70)
print("D. CONTENT COUNT FEATURES from get_counts()")
print("="*70)
content_count_cols = [c for c in cnts.columns if c.endswith('_COUNT')]
print(f"Content count columns ({len(content_count_cols)}):")
for c in sorted(content_count_cols):
    total = cnts[c].sum()
    nonzero = (cnts[c] > 0).sum()
    print(f"  {c:45s}  total={total:>10,.0f}  nonzero_rows={nonzero:>7,}")

print("\n" + "="*70)
print("E. CONTENT TIMEFRAME FEATURES from process_timeframes()")
print("="*70)
content_tf_cols = [c for c in result_df.columns if '_COUNT_' in c]
print(f"Content × timeframe columns ({len(content_tf_cols)}):")
for window in ['C1', 'C3', 'P3', 'P6', 'P12']:
    w_cols = [c for c in content_tf_cols if c.endswith(f'_{window}')]
    print(f"\n  {window} window ({len(w_cols)} cols):")
    for c in sorted(w_cols):
        total = result_df[c].sum()
        print(f"    {c:55s}  total={total:>10,.0f}")

print("\n" + "="*70)
print("F. CONTENT SEQUENCE FEATURES from SequenceProcessor")
print("="*70)
content_seq_cols = [
    'full_contentName_seq', 'full_contentCode_seq',
    'contentName_seq', 'contentCode_seq',
    'full_contentName_seq_GAP_grp', 'full_contentCode_seq_GAP_grp'
]
print(f"Content sequence columns: {content_seq_cols}")
print(f"final_df shape: {final_df.shape}")
if len(final_df) > 0:
    print("\nSample content sequences (first 3 rows):")
    display(final_df[['CUSTOMER_ID'] + [c for c in content_seq_cols if c in final_df.columns]].head(3))
else:
    print("(SequenceProcessor returned 0 rows for this instance date range — sequences are empty)")

print("\n" + "="*70)
print("SUMMARY: Total downstream features from Content_tagging_6_5.xlsx")
print("="*70)
print(f"  Content count columns (per YRMO):        {len(content_count_cols)}")
print(f"  Content × timeframe columns:             {len(content_tf_cols)}")
print(f"  Content sequence columns:                {len(content_seq_cols)}")
print(f"  TOTAL content-tagging derived features:  {len(content_count_cols) + len(content_tf_cols) + len(content_seq_cols)}")

In [0]:
# ===== SECTION 3.5 VERIFICATION =====
# Family 5: Engagement depth and content
# Required: R/I/E per placement for 5 vendors + content-category counts + iVA
# Source: kerendia_oe_journey.oe_kerendia_fact_npp_inference (the table we already trace)

print("="*80)
print("SECTION 3.5 — FAMILY 5: ENGAGEMENT DEPTH AND CONTENT")
print("Mapping OE pipeline output → NBA 3.1 requirements")
print("="*80)

# --- A. FIVE TARGET VENDORS ---
# Section 3.5 says: Doximity, Medscape, PeerDirect, PulsePoint, SFMC
target_vendors = ['DOXIMITY', 'MEDSCAPE', 'PEERDIRECT', 'PULSEPOINT', 'SFMC']

print("\n" + "-"*80)
print("A. R/I/E COUNTS PER PLACEMENT (5 target vendors)")
print("-"*80)

# Extract vendor from CHANNEL_ID: format is VENDOR_PLACEMENT or VENDOR_PLACEMENT_METRIC
all_R_cols = [c for c in cnts.columns if c.endswith('_R')]
all_I_cols = [c for c in cnts.columns if c.endswith('_I')]
all_E_cols = [c for c in cnts.columns if c.endswith('_E')]

print(f"\nTotal R/I/E columns from get_counts(): R={len(all_R_cols)}, I={len(all_I_cols)}, E={len(all_E_cols)}")

# Classify each R/I/E column by vendor
for vendor in target_vendors:
    r_match = [c for c in all_R_cols if c.startswith(vendor)]
    i_match = [c for c in all_I_cols if c.startswith(vendor)]
    e_match = [c for c in all_E_cols if c.startswith(vendor)]
    total_v = len(r_match) + len(i_match) + len(e_match)
    print(f"\n  {vendor}: {total_v} features (R={len(r_match)}, I={len(i_match)}, E={len(e_match)})")
    for suffix, cols in [('R', r_match), ('I', i_match), ('E', e_match)]:
        for c in sorted(cols):
            total = cnts[c].sum()
            nz = (cnts[c] > 0).sum()
            print(f"    {c:55s} total={total:>10,.0f}  nonzero={nz:>6,}")

# Non-target channels (personal promo — not in section 3.5)
other_R = [c for c in all_R_cols if not any(c.startswith(v) for v in target_vendors)]
other_I = [c for c in all_I_cols if not any(c.startswith(v) for v in target_vendors)]
other_E = [c for c in all_E_cols if not any(c.startswith(v) for v in target_vendors)]
print(f"\n  NON-TARGET (personal promo, NOT in 3.5): R={len(other_R)}, I={len(other_I)}, E={len(other_E)}")
for c in sorted(other_R + other_I + other_E):
    print(f"    {c}")

# --- B. CONTENT CATEGORY COUNTS ---
print("\n" + "-"*80)
print("B. CONTENT-CATEGORY COUNTS")
print("-"*80)
print(f"Section 3.5 says: 13 categories + GENERAL = 14 expected")
print(f"OE pipeline produces: {len(content_count_cols)} content count columns")
print()
for c in sorted(content_count_cols):
    tag = c.replace('_COUNT', '')
    total = cnts[c].sum()
    print(f"  {tag:40s} → {c:45s} total={total:>10,.0f}")

# Check which come from content tagging vs hardcoded "General"
email_only_tags = [t for t in proto_idc['CONTENT_TAG'].unique() if t != 'GENERAL']
print(f"\nTags from content tagging (email only, non-GENERAL): {len(email_only_tags)}")
print(f"Tags: {sorted(email_only_tags)}")

# --- C. TIMEFRAME AGGREGATION ---
print("\n" + "-"*80)
print("C. TIMEFRAME FEATURES (R/I/E × time window)")
print("-"*80)

# R/I/E timeframe features
rie_tf_cols = [c for c in result_df.columns 
               if any(c.endswith(f'_{w}') for w in ['C1','C3','P3','P6','P12'])
               and not c.endswith('_COUNT_C1') and not c.endswith('_COUNT_C3')
               and not c.endswith('_COUNT_P3') and not c.endswith('_COUNT_P6')
               and not c.endswith('_COUNT_P12')
               and ('_R_' in c or '_I_' in c or '_E_' in c or c.endswith('_R') or c.endswith('_I') or c.endswith('_E'))]

# Actually let's classify all timeframe cols properly
all_tf_cols = [c for c in result_df.columns 
               if any(c.endswith(f'_{w}') for w in ['C1','C3','P3','P6','P12'])]
rie_tf = [c for c in all_tf_cols if any(c.endswith(f'_{m}_{w}') for m in ['R','I','E'] for w in ['C1','C3','P3','P6','P12'])]
content_tf = [c for c in all_tf_cols if '_COUNT_' in c]
other_tf = [c for c in all_tf_cols if c not in rie_tf and c not in content_tf]

print(f"R/I/E × timeframe columns:       {len(rie_tf)}")
print(f"Content × timeframe columns:     {len(content_tf)}")
print(f"Other timeframe columns:         {len(other_tf)}")
if other_tf:
    print(f"  Other: {other_tf[:10]}")

# Vendor breakdown of R/I/E timeframe
for vendor in target_vendors:
    v_cols = [c for c in rie_tf if vendor in c]
    print(f"  {vendor}: {len(v_cols)} R/I/E × timeframe features")

# --- D. SECTION 3.5 MAPPING TABLE ---
print("\n" + "-"*80)
print("D. NBA 3.1 SECTION 3.5 → OE PIPELINE MAPPING")
print("-"*80)

sec35_features = {
    'R/I/E per placement (5 vendors)': {
        'expected': '~30-40 (5 vendors × placements × R/I/E × windows)',
        'example': 'DIGITAL_EVT_DOXIMITY_DOCSPOT_E_3M',
    },
    'Content-category counts': {
        'expected': '13 categories + GENERAL = 14 (× windows = ~70)',
        'example': 'content_tag_COUNT per YRMO',
    },
    'iVA campaign features': {
        'expected': 'TBD (campaign and content IDs do not exist yet)',
        'example': 'N/A',
    },
}

# Count what we actually produce for target vendors only
target_rie_monthly = sum(1 for c in all_R_cols + all_I_cols + all_E_cols 
                         if any(c.startswith(v) for v in target_vendors))
target_rie_tf = sum(1 for c in rie_tf if any(v in c for v in target_vendors))

print(f"\n{'Requirement':45s} | {'Expected':25s} | {'OE Pipeline Produces':30s} | Status")
print("-"*135)
print(f"{'R/I/E per placement (5 vendors, monthly)':45s} | {'~30-40 columns':25s} | {target_rie_monthly:>3d} columns{' ':21s} | {'✅ YES' if target_rie_monthly > 0 else '❌ NO'}")
print(f"{'R/I/E per placement × timeframes':45s} | {'~150-200 columns':25s} | {target_rie_tf:>3d} columns{' ':21s} | {'✅ YES' if target_rie_tf > 0 else '❌ NO'}")
print(f"{'Content-category counts (monthly)':45s} | {'14 columns':25s} | {len(content_count_cols):>3d} columns{' ':21s} | {'✅ YES' if len(content_count_cols) >= 14 else '⚠️ PARTIAL'}")
print(f"{'Content × timeframe features':45s} | {'~70 columns':25s} | {len(content_tf):>3d} columns{' ':21s} | {'✅ YES' if len(content_tf) > 0 else '❌ NO'}")
print(f"{'iVA campaign features':45s} | {'TBD':25s} | {'  0 columns':30s} | {'⏳ NOT YET (expected)'}")

# --- E. GAPS AND ISSUES ---
print("\n" + "-"*80)
print("E. GAPS vs SECTION 3.5 SPEC")
print("-"*80)

gaps = []

# Gap 1: Doctor key
print("\n1. DOCTOR KEY")
print(f"   Section 3.5 requires: NPI")
print(f"   OE pipeline uses:     CUSTOMER_ID (= BH_ID / CUST_ID)")
print(f"   → NEEDS: join through kerendia_oe_npi_bhid_mapping_inference")
gaps.append("BH_ID→NPI mapping needed")

# Gap 2: Personal channels mixed in
print("\n2. CHANNEL SCOPE")
print(f"   Section 3.5 requires: 5 vendors only (Doximity, Medscape, PeerDirect, PulsePoint, SFMC)")
personal_channels = sorted(set(other_R + other_I + other_E))
print(f"   OE pipeline also includes: {len(personal_channels)} personal promo channels")
print(f"   Personal channels: {[c.split('_')[0] for c in other_R]}")
print(f"   → NEEDS: filter to 5 vendors, or separate personal vs non-personal")
gaps.append("Filter to 5 target vendors (exclude personal promo channels)")

# Gap 3: Window definition  
print("\n3. TIME WINDOWS")
print(f"   Section 3.5 requires: 1, 3, 6, 12 complete months before anchor date")
print(f"   OE pipeline uses:     C1, C3, P3, P6, P12 (different window logic)")
print(f"   → CHECK: verify C1=1M, C3=3M, P3=prior3M, etc. align with 3.5 spec")
gaps.append("Verify window definitions match section 3.5")

# Gap 4: Content categories count
print("\n4. CONTENT CATEGORIES")
print(f"   Section 3.5 says: 13 categories + GENERAL = 14")
print(f"   OE pipeline produces: {len(content_count_cols)} categories (including GENERAL)")
extra = len(content_count_cols) - 14
if extra > 0:
    print(f"   → {extra} extra tag(s) vs the spec")
elif extra < 0:
    print(f"   → {-extra} fewer tag(s) than the spec")
else:
    print(f"   → EXACT MATCH")

# Gap 5: Naming convention
print("\n5. NAMING CONVENTION")
print(f"   Section 3.5 example: DIGITAL_EVT_DOXIMITY_DOCSPOT_E_3M")
sample_cols = [c for c in all_R_cols if c.startswith('DOXIMITY')][:2]
print(f"   OE pipeline example: {sample_cols}")
print(f"   → Rename needed for NBA 3.1 schema alignment")
gaps.append("Rename features to NBA 3.1 naming convention")

# Gap 6: iVA
print("\n6. iVA CAMPAIGN")
print(f"   Section 3.5 says: add iVA once campaign and content IDs exist")
print(f"   OE pipeline: not built yet (expected)")

# Gap 7: Sequences are empty
print("\n7. SEQUENCE FEATURES")
print(f"   SequenceProcessor produced 0 rows for instance-1 date range")
print(f"   Sequences are an OE-internal feature, NOT required by section 3.5")
print(f"   → Not a gap (sequences are for OE model, not NBA 3.1)")

print("\n" + "="*80)
print("VERDICT")
print("="*80)
print(f"""\nThe OE pipeline (data_processing.py → generate_mdb) DOES produce the
core section 3.5 features:

  ✅ R/I/E counts per placement for all 5 target vendors
  ✅ Content-category counts ({len(content_count_cols)} tags)
  ✅ Timeframe aggregations (C1/C3/P3/P6/P12)

Gaps to close before using as NBA 3.1 Family 5 features:
""")
for i, g in enumerate(gaps, 1):
    print(f"  {i}. {g}")
print(f"\n  Total feature count from 5 target vendors: {target_rie_monthly} R/I/E monthly + {len(content_count_cols)} content = {target_rie_monthly + len(content_count_cols)} per YRMO")
print(f"  With timeframe expansion: {target_rie_tf} R/I/E + {len(content_tf)} content = {target_rie_tf + len(content_tf)} features")
print(f"  Section 3.5 target: 40 to 60 features")

## Cohort Features (NBA 3.1 promotional cohorts)

1. **HCP x Date** — from `proto_idc` (Step 4): daily counts per HCP for every `{CHANNEL}_{MAPPED_METRIC}_{R|I|E}` (R on `EXPOSED_ON`, I on `ENGAGED_ON`, E on `ACTED_ON`), every `{CONTENT_TAG}_COUNT` (on `EXPOSED_ON`), and the 4 touchpoint roll-ups of `process_timeframes()`.
2. **HCP x Cohort** — daily rows are summed over windows that end on the cohort `lookback_end_date`: `P{w}` = (lookback_end_date − w months, lookback_end_date] for w = 1, 3, 6, 12. `P12` is exactly the cohort lookback (lookback_start_date, lookback_end_date]. No event after `lookback_end_date` is used.
3. Every HCP in the modelling universe (`df_target`) gets one row per cohort (0 = no activity). `FULL_DATA_COVERAGE` = 0 when the lookback starts before the first date in the source extracts.

In [0]:
# ===== COHORT FEATURES 1: HCP x DATE level =====
# Input : proto_idc from Step 4 (calls + emails + events, standardized, deduped). Its window is the union of all
#         cohort lookbacks (START_YRMO / END_YRMO are set in the cohort cell).
# Output: hcp_date_long     (Spark, long)  CUSTOMER_ID x EVENT_DATE x FEATURE -> COUNT
#         hcp_date_features (Spark, wide)  one row per CUSTOMER_ID x EVENT_DATE
# Feature names follow get_counts(): {CHANNEL}_{MAPPED_METRIC}_{R|I|E} and {CONTENT_TAG}_COUNT,
# plus the 4 touchpoint roll-ups of process_timeframes().

# Step 4 shifted every date by +PROMO_LAG_DAYS. The promotional cohorts already carry a 17-day offset,
# so by default the shift is undone and features use the actual event date.
KEEP_STEP4_LAG = False
UNDO_DAYS = 0 if KEEP_STEP4_LAG else PROMO_LAG_DAYS

DATE_COL = {'R': 'EXPOSED_ON', 'I': 'ENGAGED_ON', 'E': 'ACTED_ON'}
base = proto_idc[['CUSTOMER_ID', 'CHANNEL_ID', 'CONTENT_TAG', 'EXPOSED_ON', 'ENGAGED_ON', 'ACTED_ON', 'COUNT']]

parts = []
for m, dc in DATE_COL.items():
    sub = base[base['CHANNEL_ID'].isin(list(metric_mapping[m].keys()))]
    parts.append(pd.DataFrame({
        'CUSTOMER_ID': sub['CUSTOMER_ID'].values,
        'EVENT_DATE':  sub[dc].values,
        'FEATURE':     (sub['CHANNEL_ID'] + '_' + sub['CHANNEL_ID'].map(metric_mapping[m]).astype(str) + '_' + m).values,
        'METRIC':      m,
        'IS_BANNER':   sub['CHANNEL_ID'].str.upper().str.contains('BANNER').values,
        'COUNT':       sub['COUNT'].values}))
ct = base['CONTENT_TAG'].replace(['NONE', None, np.nan, 'NAN', ''], 'GENERAL')
parts.append(pd.DataFrame({
    'CUSTOMER_ID': base['CUSTOMER_ID'].values, 'EVENT_DATE': base['EXPOSED_ON'].values,
    'FEATURE': (ct + '_COUNT').values, 'METRIC': 'CONTENT', 'IS_BANNER': False, 'COUNT': base['COUNT'].values}))
long_pd = pd.concat(parts, ignore_index=True)

long_pd['EVENT_DATE'] = pd.to_datetime(long_pd['EVENT_DATE'], errors='coerce') - pd.Timedelta(days=UNDO_DAYS)
# nan_handler filled missing ENGAGED_ON / ACTED_ON with 1970-01-01 = "no engagement / action" -> drop
n_before = len(long_pd)
long_pd = long_pd[long_pd['EVENT_DATE'] >= pd.Timestamp('2000-01-01')]
print(f"Dropped {n_before - len(long_pd):,} rows with no event date (1970 fill / NaT)")
long_pd = long_pd.groupby(['CUSTOMER_ID', 'EVENT_DATE', 'FEATURE', 'METRIC', 'IS_BANNER'], as_index=False)['COUNT'].sum()
long_pd['EVENT_DATE'] = long_pd['EVENT_DATE'].dt.strftime('%Y-%m-%d')
long_pd['COUNT'] = long_pd['COUNT'].astype('int64')
long_pd['IS_BANNER'] = long_pd['IS_BANNER'].astype(bool)

hcp_date_long = spark.createDataFrame(long_pd).withColumn("EVENT_DATE", F.to_date("EVENT_DATE"))

# Touchpoint roll-ups (same definitions as process_timeframes)
TOUCHPOINTS = [
    ("EXPOSURE_TOUCHPOINT", col("METRIC") == "R"),
    ("EXPOSURE_TOUCHPOINT_BANNER_ENGAGEMENT", ((col("METRIC") == "R") & ~col("IS_BANNER")) | ((col("METRIC") == "I") & col("IS_BANNER"))),
    ("ENGAGEMENT_TOUCHPOINT", col("METRIC") == "I"),
    ("INTERACTION_TOUCHPOINT", col("METRIC") == "E"),
]
tp_rows = reduce(lambda a, b: a.unionByName(b), [
    hcp_date_long.filter(cond).groupBy("CUSTOMER_ID", "EVENT_DATE").agg(F.sum("COUNT").alias("COUNT"))
                 .withColumn("FEATURE", lit(name)) for name, cond in TOUCHPOINTS])
hcp_date_long = (hcp_date_long.select("CUSTOMER_ID", "EVENT_DATE", "FEATURE", "COUNT")
                 .unionByName(tp_rows.select("CUSTOMER_ID", "EVENT_DATE", "FEATURE", "COUNT")))

FEATURES = sorted(long_pd['FEATURE'].unique().tolist()) + [n for n, _ in TOUCHPOINTS]
hcp_date_features = (hcp_date_long.groupBy("CUSTOMER_ID", "EVENT_DATE")
                     .pivot("FEATURE", FEATURES).agg(F.sum("COUNT")).fillna(0))

rng = hcp_date_long.agg(F.min("EVENT_DATE"), F.max("EVENT_DATE"), F.countDistinct("CUSTOMER_ID")).first()
print(f"HCP x date features: {hcp_date_features.count():,} rows x {len(FEATURES)} features")
print(f"Event dates {rng[0]} .. {rng[1]}, HCPs with activity: {rng[2]:,}")
print("Features:", FEATURES)
display(hcp_date_features.orderBy(F.desc("EVENT_DATE"), "CUSTOMER_ID").limit(10))

In [0]:
# ===== COHORT FEATURES 2: HCP x DATE -> HCP x COHORT =====
# P{w} = (lookback_end_date - w months, lookback_end_date]; P12 = the full cohort lookback. Nothing after lookback_end_date is used.
WINDOWS = [1, 3, 6, 12]

import builtins  # `min` is shadowed by a pyspark star-import in an earlier cell
src_min = builtins.min(d.agg(F.min("DATE_ID")).first()[0] for d in [df_call_src, df_npp_src, df_evnt_src])
DATA_START = pd.to_datetime(src_min, format="%Y%m%d").date()

coh = cohort_anchor_date.select("cohort_id", "prediction_month", "dataset", "offset_days",
                                "lookback_start_date", "lookback_end_date")
for w in WINDOWS:
    coh = coh.withColumn(f"P{w}_START", F.add_months("lookback_end_date", -w))
assert coh.filter(col("P12_START") != col("lookback_start_date")).count() == 0, "P12 must equal the cohort lookback"
coh = coh.withColumn("FULL_DATA_COVERAGE", (col("lookback_start_date") >= lit(DATA_START)).cast("int"))

j = hcp_date_long.join(F.broadcast(coh), (col("EVENT_DATE") > col("lookback_start_date")) &
                                         (col("EVENT_DATE") <= col("lookback_end_date")))
win = F.array(*[F.when(col("EVENT_DATE") > col(f"P{w}_START"), lit(w)) for w in WINDOWS])
agg_long = (j.withColumn("W", F.explode(win)).filter(col("W").isNotNull())
             .groupBy("CUSTOMER_ID", "cohort_id", "FEATURE", "W").agg(F.sum("COUNT").alias("COUNT"))
             .withColumn("FEAT_W", F.concat(col("FEATURE"), lit("_P"), col("W").cast("string"))))

FEAT_COLS = [f"{f}_P{w}" for f in FEATURES for w in WINDOWS]
cohort_feats = agg_long.groupBy("CUSTOMER_ID", "cohort_id").pivot("FEAT_W", FEAT_COLS).agg(F.first("COUNT"))

# Every HCP in the modelling universe gets one row per cohort (0 = no activity in the window)
universe = df_target.select(col("BH_ID").alias("CUSTOMER_ID")).distinct()
hcp_cohort_features = (universe
    .crossJoin(F.broadcast(coh.select("cohort_id", "prediction_month", "dataset",
                                      "lookback_start_date", "lookback_end_date", "FULL_DATA_COVERAGE")))
    .join(cohort_feats, ["CUSTOMER_ID", "cohort_id"], "left")
    .fillna(0, subset=FEAT_COLS))

print(f"Source data starts {DATA_START}; cohorts with full lookback coverage: "
      f"{[r.cohort_id for r in coh.filter(col('FULL_DATA_COVERAGE') == 1).orderBy('lookback_end_date').collect()]}")
print(f"HCP x cohort features: {hcp_cohort_features.count():,} rows, {len(FEAT_COLS)} feature columns")

display(hcp_cohort_features.groupBy("cohort_id", "dataset", "lookback_start_date", "lookback_end_date", "FULL_DATA_COVERAGE")
        .agg(F.count("*").alias("HCPS"),
             F.sum((col("EXPOSURE_TOUCHPOINT_P12") > 0).cast("int")).alias("HCPS_EXPOSED_P12"),
             F.sum("EXPOSURE_TOUCHPOINT_P3").alias("EXPOSURE_P3"),
             F.sum("EXPOSURE_TOUCHPOINT_P12").alias("EXPOSURE_P12"),
             F.sum("ENGAGEMENT_TOUCHPOINT_P12").alias("ENGAGEMENT_P12"),
             F.sum("INTERACTION_TOUCHPOINT_P12").alias("INTERACTION_P12"))
        .orderBy("lookback_end_date"))

In [0]:
# ===== COHORT FEATURES 3: QC =====
qc = []
def check(name, ok, detail=""):
    qc.append((name, "PASS" if ok else "FAIL", str(detail)))

n_rows = hcp_cohort_features.count()
n_keys = hcp_cohort_features.select("CUSTOMER_ID", "cohort_id").distinct().count()
n_exp = universe.count() * coh.count()
check("QC1 one row per HCP x cohort", n_rows == n_keys == n_exp, f"{n_rows:,} rows, {n_keys:,} keys, expected {n_exp:,}")

# QC2 nested windows: P1 <= P3 <= P6 <= P12 for every feature
nest = reduce(lambda a, b: a | b, [col(f"{f}_P{a}") > col(f"{f}_P{b}") for f in FEATURES for a, b in zip(WINDOWS, WINDOWS[1:])])
n_bad = hcp_cohort_features.filter(nest).count()
check("QC2 nested windows P1<=P3<=P6<=P12", n_bad == 0, f"{n_bad:,} violating rows")

# QC3 activity outside the universe (dropped by the left join)
n_out = hcp_date_long.select("CUSTOMER_ID").distinct().join(universe, "CUSTOMER_ID", "left_anti").count()
check("QC3 active HCPs outside df_target", n_out == 0, f"{n_out:,} HCPs")

# QC4 independent recompute straight from proto_idc (pandas) for 5 HCPs in the latest cohort
last = coh.orderBy(col("lookback_end_date").desc()).first()
end = pd.Timestamp(last["lookback_end_date"])
raw_r = proto_idc[proto_idc['CHANNEL_ID'].isin(list(metric_mapping['R'].keys()))].copy()
raw_r['D'] = pd.to_datetime(raw_r['EXPOSED_ON']) - pd.Timedelta(days=UNDO_DAYS)
sample_rows = (hcp_cohort_features.filter((col("cohort_id") == last["cohort_id"]) & (col("EXPOSURE_TOUCHPOINT_P3") > 0))
               .select("CUSTOMER_ID", "EXPOSURE_TOUCHPOINT_P3", "EXPOSURE_TOUCHPOINT_P12")
               .orderBy("CUSTOMER_ID").limit(5).collect())
recompute = []
for r in sample_rows:
    s = raw_r[raw_r['CUSTOMER_ID'] == r.CUSTOMER_ID]
    m3 = int(s[(s.D > end - pd.DateOffset(months=3)) & (s.D <= end)]['COUNT'].sum())
    m12 = int(s[(s.D > end - pd.DateOffset(months=12)) & (s.D <= end)]['COUNT'].sum())
    recompute.append((r.CUSTOMER_ID, last["cohort_id"], int(r.EXPOSURE_TOUCHPOINT_P3), m3,
                      int(r.EXPOSURE_TOUCHPOINT_P12), m12))
mism = [x for x in recompute if x[2] != x[3] or x[4] != x[5]]
check("QC4 manual recompute EXPOSURE_TOUCHPOINT P3/P12 (5 HCPs)", len(sample_rows) > 0 and not mism, f"{len(mism)} mismatches of {len(sample_rows)}")

display(pd.DataFrame(qc, columns=["CHECK", "RESULT", "DETAIL"]))
display(pd.DataFrame(recompute, columns=["CUSTOMER_ID", "COHORT", "PIPE_P3", "MANUAL_P3", "PIPE_P12", "MANUAL_P12"]))